# Reproduction pipeline — Benchmarking quantum machine learning against classical models for the diagnostic classification of obstructive coronary artery disease in the UCI Heart Disease cohorts

This notebook reproduces **every number, table and figure** of the article from the original UCI Heart Disease files,
then checks the results against the reported values and runs an independent audit.

**How to run in Google Colab:** *Runtime → Run all*. If the installation cell reports that NumPy/pandas were changed,
choose *Runtime → Restart session* once and then *Run all* again (the installation cell is skipped when the pinned
versions are already present).

| Step | Content | Approx. time (1 CPU core) |
|---|---|---|
| 0–2 | Pinned installation, data download with SHA-256 check | 2–4 min |
| A | Cohorts, data-provenance audit, odds ratios, 6 classical models | 1 min |
| B | VQC-DR, VQC-3L, VQC-TE, QSVM, LR-PCA4, XGBoost-PCA4; seeds; 200 epochs | 8 min |
| D | Conformal prediction, gradient variance, gate-level noise (VQC-DR), subgroups, decision curves | 6 min |
| E | Ablation (15 epochs, 5 seeds) | 10 min |
| F | Gate-level noise for VQC-TE | 2 min |
| G | Paired bootstrap AUC differences | 3 min |
| C | Learning curves on identical sub-samples | 15 min |
| Figures 1–8 and S1–S3, tables 1–2 and S1–S7 (CSV and Excel), verification against the article, independent audit | | 6 min |

Set `RUN_LEARNING_CURVES = False` in the configuration cell for a faster run (step C and its checks are then skipped).

In [ ]:
# Configuration
RUN_LEARNING_CURVES = True     # step C (~15 min)
RUN_AUDIT = True               # independent re-implementation checks (~5 min)
WORKDIR = 'qml_heart_repro'
import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR); print('working directory:', os.getcwd())

## 0. Pinned software environment

In [ ]:
import importlib, subprocess, sys
PINS = {'numpy': '2.4.4', 'pandas': '3.0.2', 'scipy': '1.17.1', 'scikit-learn': '1.8.0', 'xgboost': '3.4.1',
        'lightgbm': '4.7.0', 'pennylane': '0.45.0', 'mapie': '1.4.1', 'statsmodels': '0.15.0', 'matplotlib': '3.10.8', 'openpyxl': '3.1.5'}
from importlib.metadata import version, PackageNotFoundError
def have(p):
    try: return version(p)
    except PackageNotFoundError: return None
missing = {p: v for p, v in PINS.items() if have(p) != v}
if missing:
    print('installing:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + [f'{p}=={v}' for p, v in missing.items()])
    if any(p in missing for p in ('numpy', 'pandas', 'scipy')):
        print('\n*** NumPy/pandas/SciPy were changed: Runtime -> Restart session, then Run all again. ***')
else:
    print('all pinned versions already present')

In [ ]:
import platform
print('Python', platform.python_version())
for p, v in PINS.items():
    got = have(p); print(f'{p:14s} required {v:8s} installed {got}', '' if got == v else '  <-- MISMATCH')
assert all(have(p) == v for p, v in PINS.items()), 'Version mismatch: restart the runtime and run all cells again.'

## 1. Data: original UCI processed files (and, for the data-provenance check only, a widely circulated third-party re-upload)

In [ ]:
import urllib.request, hashlib
os.makedirs('data', exist_ok=True)
BASE = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/'
FILES = {
 'processed.cleveland.data':   (BASE + 'processed.cleveland.data',   'a74b7efa387bc9d108d7d0115d831fe9b414b29ae7124f331b622b4efa0427c8'),
 'processed.hungarian.data':   (BASE + 'processed.hungarian.data',   'd1ad108f785768cd3d7e82dc522e6f5a61eea93cccfb3a46ee8076f73fc3d796'),
 'processed.switzerland.data': (BASE + 'processed.switzerland.data', '834a405ccf5b66ab4056bb77794adc8df0b7125186454c0a1d002d33c6c3b314'),
 'processed.va.data':          (BASE + 'processed.va.data',          'e7c93d8d0d2acdadfa4c5e8de768e2191e7f618b952e29623f1f0d5949ff6b8f'),
 'kaggle_heart.csv': ('https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv',
                      '7c3014365675306819510a49ff289efbec1d1a6a666a2dc7652f1547b383d859'),
}
for f, (url, sha) in FILES.items():
    path = os.path.join('data', f)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    h = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print(f'{f:28s}', 'SHA-256 OK' if h == sha else f'SHA-256 DIFFERS ({h})')
    assert h == sha, f'{f}: file differs from the one used in the manuscript'

## 2. Analysis code (each script is written to the working directory and run in section 3)

In [ ]:
%%writefile common.py
"""Shared utilities: data loading, leakage-free preprocessing, metrics and bootstrap.

Data: original UCI Heart Disease 'processed' files (dataset 45). A widely circulated
third-party re-upload of the Cleveland records is NOT used for modelling: its outcome
label is inverted and its ca/thal columns contain undefined codes for missing values.
"""
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             recall_score, brier_score_loss, confusion_matrix)
import statsmodels.api as sm

import os
DATA = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'data')
COLS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
        'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
FEAT = COLS[:-1]
FEAT10 = [f for f in FEAT if f not in ('slope', 'ca', 'thal')]
SEED = 42


def load(site):
    d = pd.read_csv(f'{DATA}/processed.{site}.data', header=None,
                    names=COLS, na_values='?')
    # physiologically impossible zeros are the source's missing-value code
    d.loc[d['chol'] == 0, 'chol'] = np.nan
    d.loc[d['trestbps'] == 0, 'trestbps'] = np.nan
    d['target'] = (d['num'] > 0).astype(int)
    return d


def xy(d, feats=FEAT):
    return d[feats].values.astype(float), d['target'].values.astype(int)


def make_pipe(clf):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()), ('clf', clf)])


class AngleScaler(BaseEstimator, TransformerMixin):
    """Min-max scaling to [0, pi] fitted on training data; external values clipped."""
    def fit(self, X, y=None):
        self.mn_ = X.min(0); self.rg_ = X.max(0) - self.mn_ + 1e-8
        return self

    def transform(self, X):
        return np.clip((X - self.mn_) / self.rg_ * np.pi, 0, np.pi)


def quantum_prep(n_comp):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('pca', PCA(n_components=n_comp, random_state=SEED)),
                     ('ang', AngleScaler())])


def amp_prep():
    """Amplitude encoding input: standardised 13 predictors padded to 16."""
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('pad', FunctionTransformer(
                         lambda X: np.hstack([X, np.zeros((len(X), 3))])))])


def calib_int_slope(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6); lp = np.log(p / (1 - p))
    try:
        slope = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]
        inter = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(),
                       offset=lp).fit().params[0]
    except Exception:
        slope, inter = np.nan, np.nan
    return float(inter), float(slope)


def metrics(y, p, thr=0.5):
    yp = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
    ci, cs = calib_int_slope(y, p)
    return dict(AUC=float(roc_auc_score(y, p)), ACC=float(accuracy_score(y, yp)),
                F1=float(f1_score(y, yp, zero_division=0)),
                SEN=float(tp / (tp + fn)) if tp + fn else np.nan,
                SPE=float(tn / (tn + fp)) if tn + fp else np.nan,
                Brier=float(brier_score_loss(y, p)), CalInt=ci, CalSlope=cs)


def boot_idx(n, y, B=1000, seed=SEED):
    rng = np.random.RandomState(seed); out = []
    while len(out) < B:
        i = rng.choice(n, n, replace=True)
        if len(np.unique(y[i])) == 2: out.append(i)
    return out


def boot_ci(y, p, idx):
    v = [roc_auc_score(y[i], p[i]) for i in idx]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))


def paired_diff(y, p1, p2, idx):
    d = [roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]) for i in idx]
    return (float(roc_auc_score(y, p1) - roc_auc_score(y, p2)),
            float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))


def classical_models():
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neural_network import MLPClassifier
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    return {
        'Logistic regression': LogisticRegression(C=1, max_iter=1000, random_state=SEED),
        'SVM-RBF': SVC(kernel='rbf', C=1, probability=True, random_state=SEED),
        'Random forest': RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=1),
        'MLP': MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, max_iter=2000, random_state=SEED),
        'XGBoost': XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                 eval_metric='logloss', random_state=SEED, verbosity=0, n_jobs=1),
        'LightGBM': LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=SEED, verbose=-1, n_jobs=1)}


In [ ]:
%%writefile quantum.py
"""Variational quantum classifiers, QSVM and training (PennyLane statevector).

Circuits are evaluated with parameter broadcasting over patients, which is
mathematically identical to evaluating each patient in a separate circuit
(verified to machine precision) but avoids Python overhead.
"""
import numpy as np, pennylane as qml, time
from sklearn.svm import SVC

_devs = {}


def dev(nq, mixed=False):
    k = (nq, mixed)
    if k not in _devs:
        _devs[k] = qml.device('default.mixed' if mixed else 'default.qubit', wires=nq)
    return _devs[k]


def ent_pairs(nq, kind):
    if kind == 'linear':
        return [(i, i + 1) for i in range(nq - 1)]
    if kind == 'circular':
        return [(i, (i + 1) % nq) for i in range(nq)] if nq > 2 else [(0, 1), (1, 0)]
    if kind == 'full':
        return [(i, j) for i in range(nq) for j in range(i + 1, nq)]
    raise ValueError(kind)


class VQC:
    """Generic re-uploading VQC.

    encoding: 'angle' (RY(x)RZ(x)), 'iqp' (H, RZ(x), ZZ(x_i x_j)), 'amplitude'
    reupload: if False the data are encoded once, followed by L variational blocks
    trainable_enc: VQC-TE learnable scales RY(a x) RZ(b x)
    """

    def __init__(self, nq=4, L=2, encoding='angle', entangle='circular',
                 reupload=True, trainable_enc=False, lr=0.08, epochs=15,
                 seed=42, fd_eps=0.10, noise=None):
        self.__dict__.update(locals()); del self.__dict__['self']
        self.n_var = 2 * nq * L
        self.n_enc = 2 * nq * L if trainable_enc else 0
        self.n_par = self.n_var + self.n_enc
        self.pairs = ent_pairs(nq, entangle)
        self._build()

    # ------------------------------------------------------------------
    def _encode(self, X, params, l):
        nq = self.nq
        if self.encoding == 'angle':
            for i in range(nq):
                if self.trainable_enc:
                    qml.RY(params[self.n_var + l * 2 * nq + 2 * i] * X[:, i], wires=i)
                    qml.RZ(params[self.n_var + l * 2 * nq + 2 * i + 1] * X[:, i], wires=i)
                else:
                    qml.RY(X[:, i], wires=i); qml.RZ(X[:, i], wires=i)
                self._n1(i); self._n1(i)
        elif self.encoding == 'iqp':
            for i in range(nq):
                qml.Hadamard(wires=i); qml.RZ(X[:, i], wires=i)
            for (i, j) in ent_pairs(nq, 'full'):
                qml.IsingZZ(X[:, i] * X[:, j], wires=[i, j])
        elif self.encoding == 'amplitude':
            qml.AmplitudeEmbedding(X, wires=range(nq), normalize=True)

    def _n1(self, i):
        if self.noise:
            qml.DepolarizingChannel(self.noise['p1'], wires=i)

    def _layer_noise(self):
        if self.noise:
            for i in range(self.nq):
                qml.AmplitudeDamping(self.noise['gamma'], wires=i)
                qml.PhaseDamping(self.noise['gamma'], wires=i)

    def _var(self, params, l):
        nq = self.nq
        for i in range(nq):
            qml.RY(params[l * 2 * nq + 2 * i], wires=i); self._n1(i)
            qml.RZ(params[l * 2 * nq + 2 * i + 1], wires=i); self._n1(i)
        for (i, j) in self.pairs:
            qml.CNOT(wires=[i, j])
            if self.noise:
                qml.DepolarizingChannel(self.noise['p2'], wires=i)
                qml.DepolarizingChannel(self.noise['p2'], wires=j)
        self._layer_noise()

    def _build(self):
        mixed = self.noise is not None

        @qml.qnode(dev(self.nq, mixed))
        def circ(X, params):
            for l in range(self.L):
                if self.encoding == 'amplitude' or not self.reupload:
                    if l == 0:
                        self._encode(X, params, 0)
                else:
                    self._encode(X, params, l)
                self._var(params, l)
            if mixed:
                return qml.probs(wires=0)
            return qml.expval(qml.PauliZ(0))
        self.circ = circ

    # ------------------------------------------------------------------
    def raw(self, X, params=None):
        params = self.params_ if params is None else params
        if self.noise is not None:
            pr = np.array(self.circ(X, params))
            p0 = pr[:, 0] if pr.ndim == 2 else np.array([pr[0]])
            ro = self.noise.get('readout', 0.0)
            p0 = p0 * (1 - ro) + (1 - p0) * ro
            shots = self.noise.get('shots')
            if shots:
                rng = np.random.RandomState(self.noise.get('shot_seed', 0))
                p0 = rng.binomial(shots, p0) / shots
            return 2 * p0 - 1
        return np.array(self.circ(X, params), dtype=float)

    @staticmethod
    def sig(z):
        return 1 / (1 + np.exp(-z))

    def predict_proba(self, X):
        return self.sig(self.raw(X))

    def fit(self, X, y, verbose=False):
        rng = np.random.RandomState(self.seed)
        p = rng.uniform(-np.pi, np.pi, self.n_var)
        if self.trainable_enc:
            p = np.concatenate([p, rng.uniform(0.8, 1.2, self.n_enc)])
        m = np.zeros(self.n_par); v = np.zeros(self.n_par)
        b1, b2, ea = 0.9, 0.999, 1e-8
        yb = y.astype(float); n = len(y)
        self.losses_ = []; t0 = time.time(); self.n_evals_ = 0
        raw = self.raw(X, p); self.n_evals_ += n
        for t in range(1, self.epochs + 1):
            prob = np.clip(self.sig(raw), 1e-7, 1 - 1e-7)
            self.losses_.append(float(-np.mean(yb * np.log(prob) + (1 - yb) * np.log(1 - prob))))
            dL = -(yb / prob - (1 - yb) / (1 - prob)) * prob * (1 - prob) / n
            g = np.zeros(self.n_par)
            for k in range(self.n_par):
                if k < self.n_var:
                    s, den = np.pi / 2, 2.0      # exact parameter-shift
                else:
                    s, den = self.fd_eps, 2 * self.fd_eps   # finite difference
                pp = p.copy(); pp[k] += s
                pm = p.copy(); pm[k] -= s
                g[k] = float(np.dot(dL, (self.raw(X, pp) - self.raw(X, pm)) / den))
                self.n_evals_ += 2 * n
            m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g ** 2
            p = p - self.lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + ea)
            raw = self.raw(X, p); self.n_evals_ += n
        prob = np.clip(self.sig(raw), 1e-7, 1 - 1e-7)
        self.final_loss_ = float(-np.mean(yb * np.log(prob) + (1 - yb) * np.log(1 - prob)))
        self.params_ = p; self.train_time_ = time.time() - t0
        return self


# ----------------------------------------------------------------------
def zz_state_fn(nq, reps=2):
    @qml.qnode(dev(nq))
    def st(X):
        for _ in range(reps):
            for i in range(nq):
                qml.Hadamard(wires=i); qml.RZ(2 * X[:, i], wires=i)
            for (i, j) in ent_pairs(nq, 'full'):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - X[:, i]) * (np.pi - X[:, j]), wires=j)
                qml.CNOT(wires=[i, j])
        return qml.state()
    return st


class QSVM:
    def __init__(self, nq=4, C=1.0, seed=42):
        self.nq, self.C, self.seed = nq, C, seed
        self.st = zz_state_fn(nq)

    def _K(self, A, B):
        return np.abs(A.conj() @ B.T) ** 2

    def fit(self, X, y):
        t0 = time.time()
        self.Str_ = np.array(self.st(X)); K = self._K(self.Str_, self.Str_)
        Y = np.where(y == 1, 1., -1.); YY = np.outer(Y, Y)
        self.kta_ = float(np.sum(K * YY) / (np.linalg.norm(K) * np.linalg.norm(YY)))
        ev = np.linalg.eigvalsh(K)
        self.min_eig_ = float(ev.min()); self.n_neg_eig_ = int((ev < -1e-10).sum())
        same = (YY > 0) & ~np.eye(len(y), dtype=bool); diff = YY < 0
        self.k_within_, self.k_between_ = float(K[same].mean()), float(K[diff].mean())
        self.svc_ = SVC(kernel='precomputed', C=self.C, probability=True,
                        random_state=self.seed).fit(K, y)
        self.train_time_ = time.time() - t0
        return self

    def predict_proba(self, X):
        S = np.array(self.st(X))
        return self.svc_.predict_proba(self._K(S, self.Str_))[:, 1]


In [ ]:
%%writefile run_a_classical.py
import json, pickle, time
from common import *
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

R = {}
cl, hu, va, sw = load('cleveland'), load('hungarian'), load('va'), load('switzerland')
X, y = xy(cl); Xe, ye = xy(hu); Xv, yv = xy(va)

# ---------------- cohort description (table S4) ----------------
def desc(d):
    o = dict(n=len(d), events=int(d.target.sum()), prev=float(d.target.mean()))
    for c in FEAT:
        s = d[c]; o[c] = dict(miss=int(s.isna().sum()), miss_pct=float(s.isna().mean()))
        if c in ('age', 'trestbps', 'chol', 'thalach', 'oldpeak'):
            o[c].update(mean=float(s.mean()), sd=float(s.std()))
        else:
            o[c].update(counts={str(int(k)): int(v) for k, v in s.value_counts().sort_index().items()})
    tot = d[FEAT].size; o['cells_missing_pct'] = float(d[FEAT].isna().sum().sum() / tot)
    o['patients_any_missing'] = int(d[FEAT].isna().any(axis=1).sum())
    return o
R['cohorts'] = {k: desc(d) for k, d in [('cleveland', cl), ('hungarian', hu), ('va', va), ('switzerland', sw)]}

# unadjusted ORs in development cohort (per SD for continuous, per coded unit otherwise)
import statsmodels.api as sm
ors = {}
for c in FEAT:
    s = cl[[c, 'target']].dropna(); x = s[c].values.astype(float)
    cont = c in ('age', 'trestbps', 'chol', 'thalach', 'oldpeak')
    if cont: x = (x - x.mean()) / x.std()
    f = sm.Logit(s.target.values, sm.add_constant(x)).fit(disp=0)
    b, se = f.params[1], f.bse[1]
    ors[c] = dict(OR=float(np.exp(b)), lo=float(np.exp(b - 1.96 * se)), hi=float(np.exp(b + 1.96 * se)),
                  p=float(f.pvalues[1]), per='SD' if cont else 'unit', n=int(len(s)))
R['unadjusted_or'] = ors

# ---------------- audit: third-party re-upload and a resampled evaluation set ----------------
kg = pd.read_csv(f'{DATA}/kaggle_heart.csv'); kg.columns = [c.strip('\ufeff') for c in kg.columns]
kg.columns = FEAT + ['target']
def make_hungarian(df_src, n=294, age_shift=3.0, seed=123):      # resampled evaluation set of an earlier analysis (audit only)
    rng = np.random.RandomState(seed); nd = int(n * 0.578); rows = []
    for label, ns in [(1, nd), (0, n - nd)]:
        src = df_src[df_src['target'] == label].copy()
        idx = rng.choice(len(src), ns, replace=True)
        s = src.iloc[idx].copy().reset_index(drop=True); s['_src'] = src.index.values[idx]
        s['age'] = np.clip(s['age'] + rng.normal(age_shift, 2., ns), 28, 80).astype(int)
        for col in ['trestbps', 'chol', 'thalach', 'oldpeak']:
            sg = s[col].std() * 0.08
            s[col] = np.clip(s[col] + rng.normal(0, sg, ns), s[col].min() * 0.9, s[col].max() * 1.1)
        s['target'] = label; rows.append(s)
    return pd.concat(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
syn = make_hungarian(kg)
cat = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
merged = syn[cat + ['target']].merge(kg[cat + ['target']].drop_duplicates(), how='left', indicator=True)
# label check: Kaggle target vs UCI num>0
key = ['age', 'sex', 'trestbps', 'chol', 'thalach', 'oldpeak']
clk = cl.copy(); m = kg.merge(clk, on=key, suffixes=('_k', '_u'))
R['first_submission_audit'] = dict(
    synthetic_n=len(syn), synthetic_events=int(syn.target.sum()),
    unique_source_records=int(syn['_src'].nunique()),
    pct_rows_derived_from_dev=1.0,
    pct_rows_with_identical_categorical_profile_in_dev=float((merged['_merge'] == 'both').mean()),
    first30_unique_sources=int(syn['_src'][:30].nunique()),
    kaggle_matched_rows=int(len(m)),
    kaggle_label_concordant=int(((m.target_k == 1) == (m.target_u == 1)).sum()),
    kaggle_label_inverted=int(((m.target_k == 1) != (m.target_u == 1)).sum()),
    kaggle_ca4=int((kg.ca == 4).sum()), kaggle_thal0=int((kg.thal == 0).sum()))
print(R['first_submission_audit'])

# ---------------- classical models ----------------
def models():
    return {
        'Logistic regression': LogisticRegression(C=1, max_iter=1000, random_state=SEED),
        'SVM-RBF': SVC(kernel='rbf', C=1, probability=True, random_state=SEED),
        'Random forest': RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=1),
        'MLP': MLPClassifier(hidden_layer_sizes=(32, 16), alpha=1e-3, max_iter=2000, random_state=SEED),
        'XGBoost': XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                 eval_metric='logloss', random_state=SEED, verbosity=0, n_jobs=1),
        'LightGBM': LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=SEED, verbose=-1, n_jobs=1)}
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
bi_e = boot_idx(len(ye), ye); bi_v = boot_idx(len(yv), yv)
C, P = {}, {'y_int': y, 'y_ext': ye, 'y_va': yv, 'folds': folds}
for name, clf in models().items():
    t0 = time.time(); cv, oof = [], np.zeros(len(y)); cvn = []
    for tr, te in folds:
        p = clone(make_pipe(clf)).fit(X[tr], y[tr]); pr = p.predict_proba(X[te])[:, 1]
        oof[te] = pr; cv.append(roc_auc_score(y[te], pr))
    # naive: preprocessing fitted on all Cleveland before splitting
    Xn = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(X))
    for tr, te in folds:
        c2 = clone(clf).fit(Xn[tr], y[tr]); cvn.append(roc_auc_score(y[te], c2.predict_proba(Xn[te])[:, 1]))
    pipe = make_pipe(clone(clf)).fit(X, y); fit_t = time.time() - t0
    pe, pv = pipe.predict_proba(Xe)[:, 1], pipe.predict_proba(Xv)[:, 1]
    # sensitivity: 10 predictors without systematic missingness
    X10, _ = xy(cl, FEAT10); Xe10, _ = xy(hu, FEAT10)
    pe10 = make_pipe(clone(clf)).fit(X10, y).predict_proba(Xe10)[:, 1]
    C[name] = dict(int_auc=float(np.mean(cv)), int_sd=float(np.std(cv)), fold_auc=[float(a) for a in cv],
                   oof_auc=float(roc_auc_score(y, oof)), delta_naive=float(np.mean(cv) - np.mean(cvn)),
                   ext=metrics(ye, pe), ext_ci=boot_ci(ye, pe, bi_e),
                   va=metrics(yv, pv), va_ci=boot_ci(yv, pv, bi_v),
                   ext10=dict(AUC=float(roc_auc_score(ye, pe10)), ci=boot_ci(ye, pe10, bi_e)),
                   time_s=fit_t)
    P[name] = dict(oof=oof, ext=pe, va=pv)
    print(f"{name:20s} CV {np.mean(cv):.3f}±{np.std(cv):.3f} naiveΔ {C[name]['delta_naive']:+.4f} "
          f"Hu {C[name]['ext']['AUC']:.3f} {C[name]['ext_ci']} VA {C[name]['va']['AUC']:.3f} Hu10 {C[name]['ext10']['AUC']:.3f}")
R['classical'] = C
json.dump(R, open('res_a.json', 'w'), indent=1)
pickle.dump(P, open('pred_a.pkl', 'wb'))


In [ ]:
%%writefile run_b_quantum.py
import json, pickle, time, sys
from common import *
from quantum import VQC, QSVM
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.base import clone

cl, hu, va = load('cleveland'), load('hungarian'), load('va')
X, y = xy(cl); Xe, ye = xy(hu); Xv, yv = xy(va)
P = pickle.load(open('pred_a.pkl', 'rb')); folds = P['folds']
bi_e = boot_idx(len(ye), ye); bi_v = boot_idx(len(yv), yv)

SPECS = {'VQC-DR': dict(L=2, lr=0.08), 'VQC-3L': dict(L=3, lr=0.06),
         'VQC-TE': dict(L=2, trainable_enc=True, lr=0.06)}
MATCHED = {'LR-PCA4': LogisticRegression(C=1, max_iter=1000, random_state=SEED),
           'XGBoost-PCA4': XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                         eval_metric='logloss', random_state=SEED, verbosity=0, n_jobs=1)}


def fit_model(name, Xtr, ytr, seed=SEED, epochs=15):
    prep = quantum_prep(4).fit(Xtr); A = prep.transform(Xtr)
    if name in SPECS:
        m = VQC(**SPECS[name], seed=seed, epochs=epochs).fit(A, ytr)
    elif name == 'QSVM':
        m = QSVM().fit(A, ytr)
    else:
        m = clone(MATCHED[name]).fit(A, ytr)
    return prep, m


def proba(prep, m, Xn):
    A = prep.transform(Xn)
    return m.predict_proba(A) if not hasattr(m, 'classes_') or isinstance(m, QSVM) else m.predict_proba(A)[:, 1]


Q = {}
for name in list(SPECS) + ['QSVM'] + list(MATCHED):
    t0 = time.time(); cv = []; oof = np.zeros(len(y))
    for tr, te in folds:
        prep, m = fit_model(name, X[tr], y[tr]); pr = proba(prep, m, X[te])
        oof[te] = pr; cv.append(roc_auc_score(y[te], pr))
    prep, m = fit_model(name, X, y)
    ptr, pe, pv = proba(prep, m, X), proba(prep, m, Xe), proba(prep, m, Xv)
    o = dict(int_auc=float(np.mean(cv)), int_sd=float(np.std(cv)), oof_auc=float(roc_auc_score(y, oof)),
             train_auc=float(roc_auc_score(y, ptr)), ext=metrics(ye, pe), ext_ci=boot_ci(ye, pe, bi_e),
             va=metrics(yv, pv), va_ci=boot_ci(yv, pv, bi_v), wall_s=time.time() - t0,
             pca_var=float(prep['pca'].explained_variance_ratio_.sum()))
    if name in SPECS:
        o.update(final_train_s=m.train_time_, n_circuit_evals=m.n_evals_, losses=m.losses_,
                 n_params=m.n_par)
    if name == 'QSVM':
        o.update(kta=m.kta_, min_eig=m.min_eig_, n_neg_eig=m.n_neg_eig_,
                 k_within=m.k_within_, k_between=m.k_between_, final_train_s=m.train_time_)
    Q[name] = o; P[name] = dict(oof=oof, ext=pe, va=pv, train=ptr)
    print(f"{name:14s} CV {o['int_auc']:.3f}±{o['int_sd']:.3f} train {o['train_auc']:.3f} "
          f"Hu {o['ext']['AUC']:.3f} {np.round(o['ext_ci'],3)} VA {o['va']['AUC']:.3f} "
          f"Brier {o['ext']['Brier']:.3f} t={o['wall_s']:.0f}s", flush=True)

# seed variability (15 epochs) and extended budget (200 epochs), full development cohort
for name in SPECS:
    aucs = []
    for s in [42, 1, 2, 3, 4]:
        prep, m = fit_model(name, X, y, seed=s); aucs.append(roc_auc_score(ye, proba(prep, m, Xe)))
    Q[name]['seed_ext_auc'] = [float(a) for a in aucs]
    prep, m = fit_model(name, X, y, epochs=200); pe2 = proba(prep, m, Xe)
    Q[name]['ext200'] = dict(AUC=float(roc_auc_score(ye, pe2)), ci=boot_ci(ye, pe2, bi_e),
                             Brier=float(brier_score_loss(ye, pe2)), final_loss=m.final_loss_,
                             train_s=m.train_time_)
    P[name]['ext200'] = pe2
    print(name, 'seeds', np.round(aucs, 3), '200ep', round(Q[name]['ext200']['AUC'], 3), flush=True)

# per-patient sequential cost of the first-submission implementation (for justification)
prep = quantum_prep(4).fit(X); A = prep.transform(X)
import pennylane as qml
dev = qml.device('default.qubit', wires=4)
@qml.qnode(dev)
def single(x, p):
    for l in range(2):
        for i in range(4): qml.RY(x[i], wires=i); qml.RZ(x[i], wires=i)
        for i in range(4): qml.RY(p[l*8+i*2], wires=i); qml.RZ(p[l*8+i*2+1], wires=i)
        for i in range(4): qml.CNOT(wires=[i, (i+1) % 4])
    return qml.expval(qml.PauliZ(0))
pr = np.random.RandomState(0).uniform(-np.pi, np.pi, 16)
t0 = time.time()
for i in range(200): single(A[i], pr)
Q['_cost'] = dict(sequential_ms_per_eval=(time.time() - t0) / 200 * 1000,
                  batched_s_per_303=None)
m = VQC(L=2); t0 = time.time()
for _ in range(20): m.raw(A, pr)
Q['_cost']['batched_s_per_303'] = (time.time() - t0) / 20
print(Q['_cost'])
json.dump(Q, open('res_b.json', 'w'), indent=1)
pickle.dump(P, open('pred_b.pkl', 'wb'))


In [ ]:
%%writefile run_d_misc.py
import json, pickle, time, warnings
warnings.filterwarnings('ignore')
from common import *
from quantum import VQC
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from xgboost import XGBClassifier
from mapie.classification import SplitConformalClassifier

cl, hu, va = load('cleveland'), load('hungarian'), load('va')
X, y = xy(cl); Xe, ye = xy(hu); Xv, yv = xy(va)
P = pickle.load(open('pred_b.pkl', 'rb')); folds = P['folds']
R = {}
def xgb():
    return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, eval_metric='logloss',
                         random_state=SEED, verbosity=0, n_jobs=1)

# ---------------- conformal prediction (LAC, alpha=0.10) ----------------
def conformalise(Xtr, ytr, seed=SEED):
    Xp, Xc, yp, yc = train_test_split(Xtr, ytr, test_size=0.25, stratify=ytr, random_state=seed)
    pipe = make_pipe(xgb()).fit(Xp, yp)
    pre = Pipeline([('imp', pipe['imp']), ('sc', pipe['sc'])])
    mp = SplitConformalClassifier(estimator=pipe['clf'], confidence_level=0.90, prefit=True)
    mp.conformalize(pre.transform(Xc), yc)
    return pre, mp, len(yp), len(yc)

def cov_stats(pre, mp, Xn, yn):
    _, s = mp.predict_set(pre.transform(Xn)); s = s[:, :, 0]
    cov = s[np.arange(len(yn)), yn]; size = s.sum(1)
    return dict(coverage=float(cov.mean()), size=float(size.mean()),
                cov_y1=float(cov[yn == 1].mean()), cov_y0=float(cov[yn == 0].mean()),
                singleton=float((size == 1).mean()), both=float((size == 2).mean()),
                empty=float((size == 0).mean()), n=int(len(yn)))

# internal: 5-fold outer CV, conformal model built inside each training fold
cov_folds = []
for tr, te in folds:
    pre, mp, _, _ = conformalise(X[tr], y[tr]); cov_folds.append(cov_stats(pre, mp, X[te], y[te]))
agg = {k: float(np.mean([c[k] for c in cov_folds])) for k in cov_folds[0] if k != 'n'}
pre, mp, n_prop, n_cal = conformalise(X, y)
R['conformal'] = dict(internal_cv=agg, internal_folds=cov_folds, n_proper=n_prop, n_cal=n_cal,
                      external=cov_stats(pre, mp, Xe, ye), va=cov_stats(pre, mp, Xv, yv),
                      apparent_full_cleveland=cov_stats(pre, mp, X, y))
# repeated calibration splits for external coverage stability
reps = []
for s in range(50):
    pre_s, mp_s, _, _ = conformalise(X, y, seed=1000 + s); reps.append(cov_stats(pre_s, mp_s, Xe, ye))
R['conformal']['external_rep50'] = {k: [float(np.mean([r[k] for r in reps])), float(np.percentile([r[k] for r in reps], 2.5)),
                                        float(np.percentile([r[k] for r in reps], 97.5))] for k in ('coverage', 'size', 'cov_y1', 'cov_y0')}
print('conformal', json.dumps(R['conformal']['internal_cv']), json.dumps(R['conformal']['external']), flush=True)

# ---------------- gradient variance (VQC-DR architecture) ----------------
GV = {}
for nq in [2, 4, 6, 8]:
    prep = quantum_prep(nq).fit(X); A = prep.transform(X)
    m = VQC(nq=nq, L=2)
    rng = np.random.RandomState(SEED); g1 = []; gall = []
    for k in range(200):
        p = rng.uniform(-np.pi, np.pi, m.n_var); x = A[rng.randint(len(A))][None, :]
        grads = []
        for j in range(m.n_var):
            pp = p.copy(); pp[j] += np.pi / 2; pm = p.copy(); pm[j] -= np.pi / 2
            grads.append((m.raw(x, pp)[0] - m.raw(x, pm)[0]) / 2)
        g1.append(grads[0]); gall.append(grads)
    gall = np.array(gall)
    GV[nq] = dict(var_theta1=float(np.var(g1, ddof=1)), mean_var_all=float(np.var(gall, axis=0, ddof=1).mean()),
                  n_init=200, n_params=m.n_var)
    print('gradvar', nq, GV[nq], flush=True)
q = np.array(list(GV)); v = np.array([GV[k]['var_theta1'] for k in GV]); va_ = np.array([GV[k]['mean_var_all'] for k in GV])
R['gradvar'] = dict(per_qubit=GV, slope_log10_theta1=float(np.polyfit(q, np.log10(v), 1)[0]),
                    slope_log10_all=float(np.polyfit(q, np.log10(va_), 1)[0]),
                    ratio_2_8_all=float(va_[0] / va_[-1]))

# ---------------- gate-level noise simulation (density matrix) ----------------
prep = quantum_prep(4).fit(X); A, Ae = prep.transform(X), prep.transform(Xe)
ideal = VQC(L=2).fit(A, y)
NZ = []
for p2 in [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10]:
    for shots in [None, 1024]:
        noise = dict(p1=p2 / 10, p2=p2, gamma=p2 / 2, readout=p2, shots=shots, shot_seed=SEED)
        m = VQC(L=2, noise=noise); m.params_ = ideal.params_
        z = m.raw(Ae); pe = VQC.sig(z)
        NZ.append(dict(p2=p2, shots=shots, AUC=float(roc_auc_score(ye, pe)), Brier=float(brier_score_loss(ye, pe)),
                       mean_abs_Z=float(np.mean(np.abs(z)))))
        print('noise', NZ[-1], flush=True)
pe_ideal = ideal.predict_proba(Ae)
# the first-submission analytical convex mixture leaves the ranking (and AUC) unchanged by construction
mix = [float(roc_auc_score(ye, (1 - p) * pe_ideal + p * 0.5)) for p in (0, 0.05, 0.2)]
R['noise'] = dict(grid=NZ, convex_mixture_auc=mix, ideal_auc=float(roc_auc_score(ye, pe_ideal)))

# ---------------- subgroups (Hungarian) and decision curves ----------------
main = ['Logistic regression', 'SVM-RBF', 'Random forest', 'MLP', 'XGBoost', 'LightGBM',
        'VQC-DR', 'VQC-3L', 'VQC-TE', 'QSVM']
pa = pickle.load(open('pred_a.pkl', 'rb'))
def pred_ext(k): return pa[k]['ext'] if k in pa else P[k]['ext']
SG = {}
groups = {'Women': hu.sex.values == 0, 'Men': hu.sex.values == 1,
          'Age <55 years': hu.age.values < 55, 'Age >=55 years': hu.age.values >= 55}
for g, mask in groups.items():
    yy = ye[mask]; bi = boot_idx(mask.sum(), yy, B=1000)
    SG[g] = dict(n=int(mask.sum()), events=int(yy.sum()))
    for k in main:
        pp = pred_ext(k)[mask]; SG[g][k] = dict(AUC=float(roc_auc_score(yy, pp)), ci=boot_ci(yy, pp, bi))
R['subgroups'] = SG
thr = np.round(np.arange(0.05, 0.71, 0.05), 2); DC = {}
prev = ye.mean()
DC['treat_all'] = [float(prev - (1 - prev) * t / (1 - t)) for t in thr]
for k in main:
    pp = pred_ext(k); nb = []
    for t in thr:
        tp = np.sum((pp >= t) & (ye == 1)); fp = np.sum((pp >= t) & (ye == 0)); n = len(ye)
        nb.append(float(tp / n - fp / n * t / (1 - t)))
    DC[k] = nb
R['dca'] = dict(thresholds=thr.tolist(), net_benefit=DC)
json.dump(R, open('res_d.json', 'w'), indent=1)
print('done D')


In [ ]:
%%writefile run_e_ablation.py
"""Ablation: every configuration trained for 15 epochs (same budget as the main
comparison) on the full Cleveland cohort and evaluated on the full Hungarian
cohort; five initialisation seeds per configuration."""
import json, warnings
warnings.filterwarnings('ignore')
from common import *
from quantum import VQC

cl, hu = load('cleveland'), load('hungarian')
X, y = xy(cl); Xe, ye = xy(hu)
SEEDS = [42, 1, 2, 3, 4]
CFG = {
    # factor A1: depth / encoding architecture
    'A1 L=1': dict(L=1), 'A1 L=2 (VQC-DR)': dict(L=2), 'A1 L=3 (VQC-3L)': dict(L=3, lr=0.06),
    'A1 L=2 trainable (VQC-TE)': dict(L=2, trainable_enc=True, lr=0.06),
    # A2 entanglement (L=2 reference otherwise)
    'A2 linear': dict(L=2, entangle='linear'), 'A2 circular': dict(L=2, entangle='circular'),
    'A2 full': dict(L=2, entangle='full'),
    # A3 qubits (PCA components = qubits)
    'A3 4 qubits': dict(L=2, nq=4), 'A3 6 qubits': dict(L=2, nq=6), 'A3 8 qubits': dict(L=2, nq=8),
    # A4 encoding
    'A4 angle': dict(L=2), 'A4 IQP': dict(L=2, encoding='iqp'), 'A4 amplitude': dict(L=2, encoding='amplitude'),
    # A5 re-uploading
    'A5 re-uploading on': dict(L=2), 'A5 re-uploading off': dict(L=2, reupload=False),
}
R = {}; cache = {}
for name, kw in CFG.items():
    full = dict(nq=4, L=2, encoding='angle', entangle='circular', reupload=True, trainable_enc=False, lr=0.08)
    full.update(kw); key = tuple(sorted(full.items()))
    if key in cache:
        R[name] = cache[key]; print(name, '(shared)', flush=True); continue
    nq = full['nq']
    prep = amp_prep() if kw.get('encoding') == 'amplitude' else quantum_prep(nq)
    prep.fit(X); A, Ae = prep.transform(X), prep.transform(Xe)
    kw2 = full
    aucs, briers, times = [], [], []
    for s in SEEDS:
        m = VQC(**kw2, seed=s).fit(A, y); pe = m.predict_proba(Ae)
        aucs.append(float(roc_auc_score(ye, pe))); briers.append(float(brier_score_loss(ye, pe)))
        times.append(m.train_time_)
    R[name] = dict(auc=aucs, mean=float(np.mean(aucs)), sd=float(np.std(aucs, ddof=1)),
                   brier_mean=float(np.mean(briers)), n_params=m.n_par, time_mean=float(np.mean(times)))
    cache[key] = R[name]
    print(f"{name:28s} {np.mean(aucs):.3f} ± {np.std(aucs, ddof=1):.3f}  {np.round(aucs, 3)}", flush=True)
json.dump(R, open('res_e.json', 'w'), indent=1)


In [ ]:
%%writefile run_f_noise.py
"""Gate-level noise simulation for the best quantum model (VQC-TE) and VQC-DR with seed 1
(sensitivity: VQC-DR seed 42 converged to a poor optimum)."""
import json, warnings
warnings.filterwarnings('ignore')
from common import *
from quantum import VQC
cl, hu = load('cleveland'), load('hungarian')
X, y = xy(cl); Xe, ye = xy(hu)
prep = quantum_prep(4).fit(X); A, Ae = prep.transform(X), prep.transform(Xe)
R = {}
for name, kw in [('VQC-TE', dict(L=2, trainable_enc=True, lr=0.06)), ('VQC-DR seed 1', dict(L=2, seed=1))]:
    ideal = VQC(**kw).fit(A, y); out = []
    for p2 in [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10]:
        for shots in [None, 1024]:
            noise = dict(p1=p2 / 10, p2=p2, gamma=p2 / 2, readout=p2, shots=shots, shot_seed=SEED)
            m = VQC(**kw, noise=noise); m.params_ = ideal.params_
            z = m.raw(Ae); pe = VQC.sig(z)
            out.append(dict(p2=p2, shots=shots, AUC=float(roc_auc_score(ye, pe)),
                            Brier=float(brier_score_loss(ye, pe)), mean_abs_Z=float(np.mean(np.abs(z)))))
            print(name, out[-1], flush=True)
    R[name] = out
json.dump(R, open('res_f.json', 'w'), indent=1)


In [ ]:
%%writefile run_g_paired.py
import json, pickle
from common import *
PA = pickle.load(open('pred_a.pkl','rb')); PB = pickle.load(open('pred_b.pkl','rb'))
ye = PB['y_ext']; yv = PB['y_va']; bi = boot_idx(len(ye), ye); bv = boot_idx(len(yv), yv)
def P(k): return PA[k] if k in PA else PB[k]
CL = ['Logistic regression','SVM-RBF','Random forest','MLP','XGBoost','LightGBM','LR-PCA4']
R = {}
for q in ['VQC-DR','VQC-3L','VQC-TE','QSVM']:
    for c in CL:
        R[f'{q} vs {c}'] = paired_diff(ye, P(q)['ext'], P(c)['ext'], bi)
        R[f'{q} vs {c} [VA]'] = paired_diff(yv, P(q)['va'], P(c)['va'], bv)
for k,v in R.items():
    if 'VQC-TE' in k or 'VQC-3L' in k: print(k, np.round(v,3))
json.dump(R, open('res_g.json','w'), indent=1)


In [ ]:
%%writefile run_c_learning.py
"""Learning curves: classical and quantum models trained on identical stratified
sub-samples of the development cohort and evaluated on the identical full
external cohort (Hungarian, n=294)."""
import json, time, warnings
warnings.filterwarnings('ignore')
from common import *
from quantum import VQC
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

cl, hu = load('cleveland'), load('hungarian')
X, y = xy(cl); Xe, ye = xy(hu)
SPECS = {'VQC-DR': dict(L=2, lr=0.08), 'VQC-3L': dict(L=3, lr=0.06),
         'VQC-TE': dict(L=2, trainable_enc=True, lr=0.06)}
SIZES = [30, 60, 120, 240, 303]; REPS = 5
R = {}
for n in SIZES:
    for r in range(REPS if n < 303 else 1):
        if n < 303:
            idx, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=100 + r)
        else:
            idx = np.arange(len(y))
        Xs, ys = X[idx], y[idx]
        for name, clf in classical_models().items():
            try:
                pe = make_pipe(clone(clf)).fit(Xs, ys).predict_proba(Xe)[:, 1]
                R.setdefault(name, {}).setdefault(str(n), []).append(float(roc_auc_score(ye, pe)))
            except Exception as e:
                print('fail', name, n, e)
        prep = quantum_prep(4).fit(Xs); A, Ae = prep.transform(Xs), prep.transform(Xe)
        pe = LogisticRegression(C=1, max_iter=1000, random_state=SEED).fit(A, ys).predict_proba(Ae)[:, 1]
        R.setdefault('LR-PCA4', {}).setdefault(str(n), []).append(float(roc_auc_score(ye, pe)))
        for name, sp in SPECS.items():
            m = VQC(**sp).fit(A, ys); pe = m.predict_proba(Ae)
            R.setdefault(name, {}).setdefault(str(n), []).append(float(roc_auc_score(ye, pe)))
            R.setdefault(name + '_time', {}).setdefault(str(n), []).append(m.train_time_)
        print(n, r, {k: round(v[str(n)][-1], 3) for k, v in R.items() if not k.endswith('_time')}, flush=True)
json.dump(R, open('res_c.json', 'w'), indent=1)


In [ ]:
%%writefile figs.py
import json, pickle, sys, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8.5,
                     'xtick.labelsize': 9, 'ytick.labelsize': 9, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300})
A = json.load(open('res_a.json')); B = json.load(open('res_b.json'))
PA = pickle.load(open('pred_a.pkl', 'rb')); PB = pickle.load(open('pred_b.pkl', 'rb'))
CL = ['Logistic regression', 'SVM-RBF', 'Random forest', 'MLP', 'XGBoost', 'LightGBM']
QM = ['VQC-DR', 'VQC-3L', 'VQC-TE', 'QSVM']
MI = ['LR-PCA4', 'XGBoost-PCA4']
OUT = 'figures/'
import os; os.makedirs(OUT, exist_ok=True)
y, ye = PB['y_int'], PB['y_ext']
def R(k): return A['classical'][k] if k in A['classical'] else B[k]
def P(k): return PA[k] if k in PA else PB[k]
COL = {'Logistic regression': '#1f77b4', 'SVM-RBF': '#ff7f0e', 'Random forest': '#2ca02c', 'MLP': '#8c564b',
       'XGBoost': '#d62728', 'LightGBM': '#9467bd', 'VQC-DR': '#17becf', 'VQC-3L': '#bcbd22',
       'VQC-TE': '#e377c2', 'QSVM': '#7f7f7f', 'LR-PCA4': '#393b79', 'XGBoost-PCA4': '#843c39'}
SHORT = {'Logistic regression': 'LR', 'Random forest': 'RF'}
def s(k): return SHORT.get(k, k)


def fig2():
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.6), gridspec_kw={'width_ratios': [1.25, 1]})
    for ax, models, title in [(axes[0], CL, 'Classical models (13 predictors)'),
                              (axes[1], QM + MI, 'Quantum and 4-component inputs')]:
        x = np.arange(len(models)); w = 0.38
        ia = [R(m)['int_auc'] for m in models]; isd = [R(m)['int_sd'] for m in models]
        ea = [R(m)['ext']['AUC'] for m in models]; ci = np.array([R(m)['ext_ci'] for m in models])
        ax.bar(x - w / 2, ia, w, yerr=isd, capsize=2.5, color='#4c72b0', label='Internal 5-fold CV (mean ± SD)')
        ax.bar(x + w / 2, ea, w, yerr=[np.array(ea) - ci[:, 0], ci[:, 1] - np.array(ea)], capsize=2.5,
               color='#dd8452', label='External, Hungarian (95% CI)')
        for xi, v in zip(x, ea):
            ax.text(xi + w / 2, 0.405, f'{v:.3f}', rotation=90, ha='center', va='bottom', fontsize=8.5, color='white')
        ax.set_xticks(x); ax.set_xticklabels([s(m) for m in models], rotation=35, ha='right')
        ax.set_ylim(0.4, 1.0); ax.axhline(0.5, ls=':', color='k', lw=0.8); ax.set_title(title, fontsize=10)
        ax.set_ylabel('AUC')
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc='upper center', ncol=2, frameon=False, fontsize=8.5)
    fig.tight_layout(rect=(0, 0, 1, 0.93)); fig.savefig(OUT + 'fig2.png'); plt.close(fig)


def fig3():
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.7))
    models = CL + ['VQC-3L', 'VQC-TE']
    for ax, key, yy, title in [(axes[0], 'oof', y, 'Internal (Cleveland, out-of-fold)'),
                               (axes[1], 'ext', ye, 'External (Hungarian, n=294)')]:
        for m in models:
            f, t, _ = roc_curve(yy, P(m)[key])
            val = R(m)['int_auc'] if key == 'oof' else R(m)['ext']['AUC']
            ax.plot(f, t, lw=1.4, color=COL[m], ls='--' if m.startswith('VQC') else '-', label=f'{s(m)} ({val:.3f})')
        ax.plot([0, 1], [0, 1], ':', color='grey', lw=0.8)
        ax.set_xlabel('1 − specificity'); ax.set_ylabel('Sensitivity'); ax.set_title(title, fontsize=10)
        ax.set_aspect('equal'); ax.legend(loc='lower right', fontsize=8.2, title='Model (AUC)', title_fontsize=8.5)
    fig.tight_layout(); fig.savefig(OUT + 'fig3.png'); plt.close(fig)


def fig8():
    models = CL + QM + MI
    mets = ['AUC', 'ACC', 'SEN', 'SPE', 'F1', 'Brier']
    M = np.array([[R(m)['ext'][k] if k != 'Brier' else 1 - R(m)['ext'][k] for k in mets] for m in models])
    fig, ax = plt.subplots(figsize=(5.6, 5.0))
    im = ax.imshow(M, cmap='RdYlGn', vmin=0.3, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(mets))); ax.set_xticklabels(['AUC', 'ACC', 'SEN', 'SPE', 'F1', '1 − Brier'])
    ax.set_yticks(range(len(models))); ax.set_yticklabels([s(m) for m in models])
    for i in range(len(models)):
        for j in range(len(mets)):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=8)
    ax.axhline(5.5, color='k', lw=1.2); ax.axhline(9.5, color='k', lw=1.2)
    for sp in ax.spines.values(): sp.set_visible(True)
    cb = fig.colorbar(im, ax=ax, fraction=0.04); cb.set_label('Metric value')
    fig.tight_layout(); fig.savefig(OUT + 'fig8.png'); plt.close(fig)


def figS_loss():
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    for m in ['VQC-DR', 'VQC-3L', 'VQC-TE']:
        L = B[m]['losses']; ax.plot(range(1, len(L) + 1), L, marker='o', ms=3, color=COL[m], label=m)
    ax.set_xlabel('Epoch (full-batch Adam step)'); ax.set_ylabel('Binary cross-entropy (training)')
    ax.legend(); fig.tight_layout(); fig.savefig(OUT + 'figS1.png'); plt.close(fig)


def figS_cal():
    fig, ax = plt.subplots(figsize=(4.8, 4.4))
    for m in ['Logistic regression', 'SVM-RBF', 'Random forest', 'XGBoost', 'VQC-3L', 'VQC-TE']:
        pt, pp = calibration_curve(ye, P(m)['ext'], n_bins=8, strategy='quantile')
        ax.plot(pp, pt, marker='o', ms=3, color=COL[m], ls='--' if m.startswith('VQC') else '-',
                label=f"{s(m)} (int {R(m)['ext']['CalInt']:.2f}, slope {R(m)['ext']['CalSlope']:.2f})".replace('-', '−'))
    ax.plot([0, 1], [0, 1], ':', color='grey'); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
    ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed proportion')
    ax.legend(fontsize=7.5, loc='upper left'); fig.tight_layout(); fig.savefig(OUT + 'figS2.png'); plt.close(fig)


if __name__ == '__main__':
    fig2(); fig3(); fig8(); figS_loss(); figS_cal()


def fig1():
    a = A['cohorts']
    fig, ax = plt.subplots(figsize=(7.2, 5.4)); ax.axis('off'); ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    def box(x, y, w, h, txt, fc='#eef3fa'):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.6', fc=fc, ec='#333333', lw=1))
        ax.text(x + w / 2, y + h / 2, txt, ha='center', va='center', fontsize=8.4)
    def arr(x1, y1, x2, y2):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle='->', lw=1))
    box(22, 88, 56, 9, 'UCI Heart Disease dataset (dataset 45), original processed files\nfour centres; n=920')
    c, h, v, s_ = a['cleveland'], a['hungarian'], a['va'], a['switzerland']
    box(1, 66, 30, 14, f"Cleveland Clinic Foundation\nn={c['n']}\nobstructive CAD {c['events']} ({100*c['prev']:.1f}%)\nDEVELOPMENT")
    box(35, 66, 30, 14, f"Hungarian Institute of Cardiology\nn={h['n']}\nobstructive CAD {h['events']} ({100*h['prev']:.1f}%)\nPRIMARY EXTERNAL VALIDATION", fc='#fdf0e6')
    box(69, 66, 30, 14, f"VA Medical Center, Long Beach\nn={v['n']}\nobstructive CAD {v['events']} ({100*v['prev']:.1f}%)\nSECONDARY EXTERNAL\nVALIDATION", fc='#fdf0e6')
    box(69, 47, 30, 11, f"University Hospital Zurich\nn={s_['n']}; only {s_['n']-s_['events']} without CAD\nand cholesterol unrecorded:\nnot used", fc='#f2f2f2')
    for x in (16, 50, 84): arr(50, 88, x, 80.5)
    ax.annotate('', xy=(99.5, 55), xytext=(78, 88.5), arrowprops=dict(arrowstyle='->', lw=1, connectionstyle='angle,angleA=0,angleB=90'))
    
    box(1, 40, 30, 19, 'Internal validation\nstratified 5-fold CV, all\npreprocessing fitted within\ntraining folds\n(classical and quantum alike)')
    box(1, 15, 30, 18, 'Final models refitted on all\n303 patients:\n6 classical (13 predictors)\n4 quantum + 2 matched-input\nclassical (4 PCA components)')
    box(35, 40, 30, 19, 'Learning curves: identical\nstratified sub-samples\n(n=30, 60, 120, 240; 5 repeats)\nand full n=303, evaluated on\nthe full Hungarian cohort')
    box(35, 15, 30, 18, 'Split conformal layer\n(XGBoost; 75/25 proper\ntraining/calibration split:\nn=227/76)')
    box(69, 15, 30, 25, 'Ablation (15 epochs, 5 seeds),\ngate-level noise simulation,\ngradient-variance analysis\n\nNo patients excluded;\nmissing values imputed with\ntraining medians')
    arr(16, 66, 16, 59.6); arr(16, 40, 16, 33.6); arr(31.6, 24, 34.4, 24)
    arr(31.6, 50, 34.4, 50)
    fig.tight_layout(); fig.savefig(OUT + 'fig1.png'); plt.close(fig)


def fig5():
    D = json.load(open('res_d.json'))['conformal']
    sets = [('Internal\nCleveland (CV)', D['internal_cv']), ('External\nHungarian', D['external']), ('External\nVA Long Beach', D['va'])]
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.9))
    x = np.arange(3); w = 0.26
    for k, (lab, key, colr) in enumerate([('All patients', 'coverage', '#4c72b0'), ('With CAD', 'cov_y1', '#c44e52'), ('Without CAD', 'cov_y0', '#55a868')]):
        vals = [d[key] for _, d in sets]
        axes[0].bar(x + (k - 1) * w, vals, w, color=colr, label=lab)
        for xi, vv in zip(x, vals): axes[0].text(xi + (k - 1) * w, 0.52, f'{vv:.2f}', ha='center', fontsize=8.5, rotation=90, color='white')
    axes[0].axhline(0.9, ls='--', color='k', lw=1); axes[0].text(-0.45, 0.91, 'target 0.90', fontsize=8.5, ha='left')
    axes[0].set_ylim(0.5, 1.12); axes[0].set_yticks([0.5,0.6,0.7,0.8,0.9,1.0]); axes[0].set_xticks(x); axes[0].set_xticklabels([l for l, _ in sets])
    axes[0].set_ylabel('Empirical coverage'); axes[0].legend(fontsize=8.5, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.0), frameon=False)
    axes[0].set_title('Marginal and class-conditional coverage', fontsize=10)
    sg = [d['singleton'] for _, d in sets]; bo = [d['both'] for _, d in sets]
    axes[1].bar(x, sg, 0.5, color='#8da0cb', label='Single-label set')
    axes[1].bar(x, bo, 0.5, bottom=sg, color='#fc8d62', label='Two-label (equivocal) set')
    for xi, (_, d) in zip(x, sets): axes[1].text(xi, 1.02, f"mean {d['size']:.2f}", ha='center', fontsize=8.5)
    axes[1].set_ylim(0, 1.12); axes[1].set_xticks(x); axes[1].set_xticklabels([l for l, _ in sets])
    axes[1].set_ylabel('Proportion of patients'); axes[1].legend(fontsize=8.5, loc='lower right')
    axes[1].set_title('Prediction-set composition', fontsize=10)
    fig.tight_layout(); fig.savefig(OUT + 'fig5.png'); plt.close(fig)


def fig7():
    G = json.load(open('res_d.json'))['gradvar']
    q = [2, 4, 6, 8]; v = [G['per_qubit'][str(k)]['mean_var_all'] for k in q]; v1 = [G['per_qubit'][str(k)]['var_theta1'] for k in q]
    fig, ax = plt.subplots(figsize=(4.8, 3.4))
    ax.semilogy(q, v, 'o-', color='#6a3d9a', label='Mean over all variational parameters')
    ax.semilogy(q, v1, 's--', color='#b15928', label='First parameter (θ₁)', ms=4)
    for a_, b_ in zip(q, v): ax.annotate(f'{b_:.4f}', (a_, b_), textcoords='offset points', xytext=(6, 5), fontsize=8)
    ax.set_xticks(q); ax.set_xlabel('Number of qubits'); ax.set_ylabel('Gradient variance (log scale)')
    ax.text(0.97, 0.95, f"slope of log10(variance): {G['slope_log10_all']:.2f} per qubit\n200 random initialisations per width",
            transform=ax.transAxes, ha='right', va='top', fontsize=7.5)
    ax.legend(fontsize=7.5, loc='lower left'); fig.tight_layout(); fig.savefig(OUT + 'fig7.png'); plt.close(fig)


def figS_dca():
    D = json.load(open('res_d.json'))['dca']; t = D['thresholds']; nb = D['net_benefit']
    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    ax.plot(t, nb['treat_all'], color='grey', lw=1, label='Treat all'); ax.axhline(0, color='k', lw=0.8, label='Treat none')
    for m in ['Logistic regression', 'SVM-RBF', 'Random forest', 'XGBoost', 'VQC-3L', 'VQC-TE']:
        ax.plot(t, nb[m], color=COL[m], ls='--' if m.startswith('VQC') else '-', label=s(m))
    ax.set_ylim(-0.1, 0.4); ax.set_xlabel('Threshold probability'); ax.set_ylabel('Net benefit')
    ax.legend(fontsize=8, ncol=2); fig.tight_layout(); fig.savefig(OUT + 'figS3.png'); plt.close(fig)


def fig4():
    L = json.load(open('res_c.json')); n = [30, 60, 120, 240, 303]
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    for j, m in enumerate(['Logistic regression', 'SVM-RBF', 'Random forest', 'MLP', 'XGBoost', 'LightGBM', 'LR-PCA4', 'VQC-DR', 'VQC-3L', 'VQC-TE']):
        mu = np.array([np.mean(L[m][str(k)]) for k in n]); sdv = np.array([np.std(L[m][str(k)], ddof=1) if len(L[m][str(k)]) > 1 else 0 for k in n])
        xs = np.array(n, float)
        if m == 'LightGBM':
            xs, mu, sdv = xs[1:], mu[1:], sdv[1:]
        ax.errorbar(xs * (1 + 0.006 * (j - 4.5)), mu, yerr=sdv, marker='o', ms=3.5, lw=1.3, capsize=2,
                    color=COL[m], ls='--' if m.startswith('VQC') else '-', label=s(m))
    ax.set_xscale('log'); ax.set_xticks(n); ax.set_xticklabels(n); ax.minorticks_off()
    ax.set_xlabel('Development training size (patients)'); ax.set_ylabel('External AUC (Hungarian, n=294)')
    ax.set_ylim(0.5, 0.95); ax.axhline(0.5, color='k', ls=':', lw=0.8)
    ax.legend(ncol=2, fontsize=7.5, loc='lower right'); fig.tight_layout(); fig.savefig(OUT + 'fig4.png'); plt.close(fig)


def fig6():
    Eb = json.load(open('res_e.json')); Fn = json.load(open('res_f.json')); Dn = json.load(open('res_d.json'))['noise']['grid']
    panels = [('A  Depth / encoding architecture', ['A1 L=1', 'A1 L=2 (VQC-DR)', 'A1 L=3 (VQC-3L)', 'A1 L=2 trainable (VQC-TE)'], ['L=1', 'L=2', 'L=3', 'L=2+TE']),
              ('B  Entanglement', ['A2 linear', 'A2 circular', 'A2 full'], ['linear', 'circular', 'full']),
              ('C  Qubit count', ['A3 4 qubits', 'A3 6 qubits', 'A3 8 qubits'], ['4', '6', '8']),
              ('D  Encoding', ['A4 angle', 'A4 IQP', 'A4 amplitude'], ['angle', 'IQP', 'amplitude']),
              ('E  Data re-uploading', ['A5 re-uploading on', 'A5 re-uploading off'], ['on', 'off'])]
    fig, axes = plt.subplots(2, 3, figsize=(7.4, 6.0)); axes = axes.ravel()
    for ax, (t, keys, labs) in zip(axes, panels):
        mu = [Eb[k]['mean'] for k in keys]; sdv = [Eb[k]['sd'] for k in keys]
        ax.bar(range(len(keys)), mu, yerr=sdv, capsize=3, color='#4c72b0', width=0.6)
        for i, (m_, s_) in enumerate(zip(mu, sdv)):
            if m_ >= 0.58: ax.text(i, 0.42, f'{m_:.3f}', ha='center', fontsize=8.5, color='white', rotation=90, va='bottom')
            else: ax.text(i, m_ + s_ + 0.015, f'{m_:.3f}', ha='center', fontsize=8.5, color='black', rotation=90, va='bottom')
        ax.set_xticks(range(len(keys))); ax.set_xticklabels(labs, fontsize=9); ax.set_ylim(0.4, 1.0)
        ax.axhline(0.5, ls=':', color='k', lw=0.8); ax.set_title(t, fontsize=10, loc='left'); ax.set_ylabel('External AUC', fontsize=9)
    ax = axes[5]
    for name, grid, colr in [('VQC-TE', Fn['VQC-TE'], COL['VQC-TE']), ('VQC-DR', Dn, COL['VQC-DR'])]:
        for shots, ls in [(None, '-'), (1024, '--')]:
            g_ = [x for x in grid if x['shots'] == shots]
            ax.plot([x['p2'] for x in g_], [x['AUC'] for x in g_], ls=ls, marker='o', ms=3, color=colr,
                    label=f"{name}, {'exact' if shots is None else '1024 shots'}")
    ax.set_xscale('symlog', linthresh=0.001); ax.set_xticks([0, 0.001, 0.01, 0.1]); ax.set_xticklabels(['0', '0.001', '0.01', '0.1'])
    ax.set_ylim(0.4, 1.0); ax.set_xlabel('Two-qubit error rate p₂', fontsize=9); ax.set_ylabel('External AUC', fontsize=9)
    ax.set_title('F  Gate-level noise', fontsize=10, loc='left'); ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(0.0, 0.53), frameon=False)
    fig.tight_layout(); fig.savefig(OUT + 'fig6.png'); plt.close(fig)


In [ ]:
%%writefile tables.py
"""Builds every table of the article (tables 1 and 2, appendix tables S1 to S7) from the result files
res_a.json ... res_g.json and writes them to tables/ as CSV files and one Excel workbook."""
import json, os, csv, numpy as np
A = json.load(open('res_a.json')); B = json.load(open('res_b.json')); D = json.load(open('res_d.json'))
E = json.load(open('res_e.json')); F = json.load(open('res_f.json')); G = json.load(open('res_g.json'))
LC = json.load(open('res_c.json')) if os.path.exists('res_c.json') else None
C = A['classical']; CO = A['cohorts']
def R(k): return C[k] if k in C else B[k]
def pct(x, d=1): return f'{100 * x:.{d}f}%'
def neg(s): return s.replace('-', '−').replace('+0.000', '0.000')
def lc_mean(model, n):
    v = LC[model][str(n)]; return float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
CL = ['Logistic regression', 'SVM-RBF', 'Random forest', 'MLP', 'XGBoost', 'LightGBM']
cl, hu, va = CO['cleveland'], CO['hungarian'], CO['va']
nzTE = F['VQC-TE']; sizes = [30, 60, 120, 240, 303]
EN = '–'
def cie(lo, hi): return f'{lo:.3f}{EN}{hi:.3f}'


# ------------------------------------------------------------------ tables 1 and 2
TAB1 = [['Model', 'Int AUC (CV), mean (SD)', 'Ext AUC (95% CI)', 'ACC', 'SEN', 'SPE', 'Brier', 'Cal int', 'Cal slope', 'LB AUC (95% CI)']]
for k in CL:
    r = C[k]; e = r['ext']
    TAB1.append([k, f"{r['int_auc']:.3f} ({r['int_sd']:.3f})", f"{e['AUC']:.3f} ({cie(*r['ext_ci'])})", f"{e['ACC']:.3f}",
                 f"{e['SEN']:.3f}", f"{e['SPE']:.3f}", f"{e['Brier']:.3f}", neg(f"{e['CalInt']:.2f}"), f"{e['CalSlope']:.2f}",
                 f"{r['va']['AUC']:.3f} ({cie(*r['va_ci'])})"])
W1 = [1350, 900, 1150, 560, 560, 560, 600, 600, 620, 1126]

TAB2 = [['Model', 'Params', 'Train AUC', 'Int AUC (CV), mean (SD)', 'Ext AUC (95% CI)', 'Seed range', 'ACC', 'SEN', 'SPE', 'Brier', 'Cal slope', 'Time (s)']]
for k, npar in [('VQC-DR', '16'), ('VQC-3L', '24'), ('VQC-TE', '32'), ('QSVM', '—'), ('LR-PCA4', '5'), ('XGBoost-PCA4', '—')]:
    r = R(k); e = r['ext']
    srange = f"{min(r['seed_ext_auc']):.3f}{EN}{max(r['seed_ext_auc']):.3f}" if 'seed_ext_auc' in r else '—'
    tt = r.get('final_train_s', None)
    TAB2.append([k, npar, f"{r['train_auc']:.3f}", f"{r['int_auc']:.3f} ({r['int_sd']:.3f})", f"{e['AUC']:.3f} ({cie(*r['ext_ci'])})",
                 srange, f"{e['ACC']:.3f}", f"{e['SEN']:.3f}", f"{e['SPE']:.3f}", f"{e['Brier']:.3f}", f"{e['CalSlope']:.2f}",
                 (f'{tt:.1f}' if tt >= 1 else '<1') if tt is not None else '<1'])
W2 = [1150, 560, 640, 850, 1120, 900, 560, 560, 560, 600, 600, 526]


# Table S1 (specification) — revised widths
S1 = [['Model', 'Type', 'Parameters and hyperparameters', 'Training', 'Uncertainty']]
S1 += [
    ['Logistic regression', 'Classical, linear', 'L2 penalty, C = 1.0, max_iter = 1000; continuous predictors linear', 'Five-fold stratified CV on Cleveland; refitted on n = 303', 'B = 1000 bootstrap'],
    ['SVM-RBF', 'Classical, kernel', 'C = 1.0, RBF kernel, Platt probability calibration', 'As above', 'As above'],
    ['Random forest', 'Classical, bagging', '500 trees; scikit-learn defaults otherwise', 'As above', 'As above'],
    ['MLP', 'Classical, neural network', 'Hidden layers 32 and 16 (ReLU); L2 0.001; Adam; max_iter = 2000', 'As above', 'As above'],
    ['XGBoost', 'Classical, boosting', 'n_estimators = 200, max_depth = 4, learning_rate = 0.1', 'As above', 'As above'],
    ['LightGBM', 'Classical, boosting', 'n_estimators = 200, learning_rate = 0.1', 'As above', 'As above'],
    ['LR-PCA4, XGBoost-PCA4', 'Classical, matched input', 'As logistic regression and XGBoost, on the four angle-scaled principal components', 'As above', 'As above'],
    ['VQC-DR', 'Quantum, variational', '4 qubits, L = 2 re-uploading blocks, 16 parameters; angle encoding RY(x)·RZ(x); circular CNOT; Adam η = 0.08; exact parameter-shift gradients', '15 full-batch epochs on n = 303; same CV folds', 'As above; five seeds'],
    ['VQC-3L', 'Quantum, variational', 'As VQC-DR with L = 3, 24 parameters; Adam η = 0.06', 'As above', 'As above'],
    ['VQC-TE', 'Quantum, variational', 'As VQC-DR with trainable encoding scales (16 encoding, 16 variational parameters; scales initialised U(0.8, 1.2)); Adam η = 0.06; finite differences (ε = 0.10) for encoding parameters', 'As above', 'As above'],
    ['QSVM', 'Quantum, kernel', 'ZZ feature map, 2 repetitions, all qubit pairs; exact fidelity kernel; classical SVM C = 1.0', 'Kernel on full training data; same CV folds', 'B = 1000 bootstrap'],
    ['Conformal layer', 'Distribution-free wrapper', 'Split conformal, LAC score, α = 0.10; XGBoost base learner as above', 'Proper training n = 227, calibration n = 76', 'Empirical coverage'],
]
WS1 = [1500, 1300, 3300, 1700, 1226]
S2 = [['Component', 'Version', 'Role'],
      ['Python', '3.12.3', 'Analysis language'],
      ['scikit-learn', '1.8.0', 'Pipelines, imputation, scaling, PCA, classical models, cross-validation, metrics'],
      ['XGBoost / LightGBM', '3.4.1 / 4.7.0', 'Gradient-boosted ensembles; XGBoost is the conformal base learner'],
      ['PennyLane', '0.45.0', 'Statevector (default.qubit) and density-matrix (default.mixed) simulation'],
      ['MAPIE', '1.4.1', 'Split conformal prediction'],
      ['NumPy / statsmodels', '2.4.4 / 0.15.0', 'Numerics; calibration intercept and slope; unadjusted odds ratios'],
      ['Simulation cost', f"{B['_cost']['sequential_ms_per_eval']:.1f} ms per patient; {1000 * B['_cost']['batched_s_per_303']:.0f} ms per 303 patients (broadcast)", 'Full 15-epoch training 4–9 s per circuit on one CPU core'],
      ['Random seeds', 'Fixed and recorded in the analysis code', 'Reproduce every split, sub-sample, initialisation and bootstrap']]
WS2 = [1800, 2600, 4626]
S3 = [['Factor', 'Levels examined', 'Reference level', 'Reported in'],
      ['A1 Depth and encoding architecture', 'L = 1; L = 2 (VQC-DR); L = 3 (VQC-3L); L = 2 with trainable encoding (VQC-TE)', 'L = 2, fixed encoding', 'Figure 6A'],
      ['A2 Entanglement structure', 'Linear; circular; full', 'Circular', 'Figure 6B'],
      ['A3 Qubit count', '4; 6; 8 (principal components = qubits)', '4 qubits', 'Figure 6C'],
      ['A4 Encoding strategy', 'Angle; IQP; amplitude (13 predictors on 4 qubits)', 'Angle', 'Figure 6D'],
      ['A5 Data re-uploading', 'Enabled; disabled (encode once, two variational blocks)', 'Enabled', 'Figure 6E'],
      ['Noise sensitivity', 'Two-qubit error p₂ ∈ {0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10}; exact or 1024 shots', 'p₂ = 0', 'Figure 6F; table S7']]
WS3 = [2300, 4000, 1500, 1226]
S3note = ('IQP, instantaneous quantum polynomial. Every configuration was trained for 15 epochs (the main-comparison budget) '
          'on the full Cleveland cohort and evaluated on the full Hungarian cohort, with five initialisation seeds; values are '
          'mean ± SD across seeds. The reference configuration is shared by factors A2 to A5.')

# Table S4 baseline characteristics
def cat(site, var, lvl):
    c = CO[site][var]; n = CO[site]['n'] - c['miss']
    v = c['counts'].get(str(lvl), 0); return f'{v} ({100 * v / n:.1f})' if n else '—'
def cont(site, var):
    c = CO[site][var]; return f"{c['mean']:.1f} ({c['sd']:.1f})" if 'mean' in c and c['miss'] < CO[site]['n'] else '—'
def miss(site, var):
    c = CO[site][var]; return f"{c['miss']} ({100 * c['miss_pct']:.1f})"
ORs = A['unadjusted_or']
def orr(v):
    o = ORs[v]; return f"{o['OR']:.2f} ({o['lo']:.2f}{EN}{o['hi']:.2f})"
S4 = [['Characteristic', 'Cleveland (n=303)', 'Hungarian (n=294)', 'Long Beach (n=200)', 'Missing, n (Cleveland / Hungarian / Long Beach)', 'Unadjusted OR (95% CI), Cleveland']]
S4.append(['Obstructive CAD, n (%)', f"{cl['events']} ({pct(cl['prev'])[:-1]})", f"{hu['events']} ({pct(hu['prev'])[:-1]})", f"{va['events']} ({pct(va['prev'])[:-1]})", '0 / 0 / 0', '—'])
rows = [('Age, years, mean (SD)', 'age', None), ('Male sex, n (%)', 'sex', 1), ('Asymptomatic chest pain (cp=4), n (%)', 'cp', 4),
        ('Resting BP, mm Hg, mean (SD)', 'trestbps', None), ('Cholesterol, mg/dL, mean (SD)', 'chol', None),
        ('Fasting glucose >120 mg/dL, n (%)', 'fbs', 1), ('Abnormal resting ECG (restecg ≥1), n (%)', 'restecg', 'ab'),
        ('Maximum heart rate, bpm, mean (SD)', 'thalach', None), ('Exercise-induced angina, n (%)', 'exang', 1),
        ('ST depression, mm, mean (SD)', 'oldpeak', None), ('Flat or downsloping ST (slope ≥2), n (%)', 'slope', 'ab2'),
        ('≥1 vessel on fluoroscopy (ca ≥1), n (%)', 'ca', 'ab'), ('Reversible or fixed defect (thal 6 or 7), n (%)', 'thal', 'ab3')]
for lab, v, lv in rows:
    vals = []
    for site in ('cleveland', 'hungarian', 'va'):
        if lv is None:
            vals.append(cont(site, v))
        else:
            c = CO[site][v]; n = CO[site]['n'] - c['miss']
            if n == 0: vals.append('—'); continue
            if lv == 'ab': k = sum(x for l, x in c['counts'].items() if int(l) >= 1)
            elif lv == 'ab2': k = sum(x for l, x in c['counts'].items() if int(l) >= 2)
            elif lv == 'ab3': k = sum(x for l, x in c['counts'].items() if int(l) in (6, 7))
            else: k = c['counts'].get(str(lv), 0)
            vals.append(f'{k} ({100 * k / n:.1f})')
    per = ' per SD' if ORs[v]['per'] == 'SD' else ' per coded unit'
    S4.append([lab] + vals + [' / '.join(miss(s_, v).split(' ')[0] for s_ in ('cleveland', 'hungarian', 'va')), orr(v) + per])
WS4 = [2300, 1250, 1250, 1250, 1500, 1476]
S4note = ('BP, blood pressure; CAD, coronary artery disease; ECG, electrocardiogram; OR, odds ratio. Percentages are of '
          'patients with a recorded value. Odds ratios are from univariable logistic regression in the development cohort, per '
          'SD for continuous predictors and per unit of the source integer coding for categorical predictors (chest pain type '
          '1–4, resting ECG 0–2, slope 1–3, ca 0–3, thal 3/6/7), as entered in the models. Zero values of cholesterol and '
          'resting blood pressure were treated as missing.')

# Table S5 learning curves
S5 = None
if LC is not None:
    S5 = [['Model', 'n=30', 'n=60', 'n=120', 'n=240', 'n=303']]
    for m in CL + ['LR-PCA4', 'VQC-DR', 'VQC-3L', 'VQC-TE']:
        row = [m]
        for n in sizes:
            mu, s_ = lc_mean(m, n)
            row.append('not estimable' if (m == 'LightGBM' and n == 30) else (f'{mu:.3f} ({s_:.3f})' if n < 303 else f'{mu:.3f}'))
        S5.append(row)
WS5 = [1800, 1445, 1445, 1445, 1445, 1446]
S5note = ('External AUC in the Hungarian cohort (n=294), mean (SD) across five stratified sub-samples of the development '
          'cohort, identical for every model; n=303 is the full cohort (single fit). Preprocessing was refitted on each '
          'sub-sample. LightGBM with its default minimum leaf size (20) cannot split at n=30 and returns a constant prediction.')

# Table S6 subgroups
SG = D['subgroups']; sgk = ['Women', 'Men', 'Age <55 years', 'Age >=55 years']
S6 = [['Model'] + [f"{g.replace('>=', '≥')} (n={SG[g]['n']}; {SG[g]['events']} events)" for g in sgk]]
for m in CL + ['VQC-DR', 'VQC-3L', 'VQC-TE', 'QSVM']:
    S6.append([m] + [f"{SG[g][m]['AUC']:.3f} ({cie(*SG[g][m]['ci'])})" for g in sgk])
WS6 = [1800, 1806, 1806, 1806, 1808]
S6note = ('External AUC (95% bootstrap CI) in the Hungarian cohort by sex and age band. With 12 events among women and 26 '
          'among patients aged ≥55 years these estimates are descriptive only and cannot support claims of equivalence or '
          'disparity.')

# Table S7 sensitivity analyses
S7 = [['Analysis', 'Model', 'Result']]
for k in CL:
    S7.append(['Without slope, ca and thal (10 predictors)', k, f"Ext AUC {C[k]['ext10']['AUC']:.3f} ({cie(*C[k]['ext10']['ci'])}) vs {C[k]['ext']['AUC']:.3f} with 13"])
for k in ['VQC-DR', 'VQC-3L', 'VQC-TE']:
    S7.append(['Five initialisation seeds (15 epochs)', k, 'Ext AUC ' + ', '.join(f'{x:.3f}' for x in B[k]['seed_ext_auc']) + ' (seeds 42, 1, 2, 3, 4)'])
for k in ['VQC-DR', 'VQC-3L', 'VQC-TE']:
    e2 = B[k]['ext200']
    S7.append(['200 training epochs (seed 42)', k, f"Ext AUC {e2['AUC']:.3f} ({cie(*e2['ci'])}); Brier {e2['Brier']:.3f}; {e2['train_s']:.0f} s"])
for q in ['VQC-TE', 'VQC-3L']:
    for c in ['Logistic regression', 'SVM-RBF', 'Random forest', 'XGBoost', 'LightGBM', 'LR-PCA4']:
        d, lo, hi = G[f'{q} vs {c}']; dv, lov, hiv = G[f'{q} vs {c} [VA]']
        S7.append(['Paired ΔAUC, quantum − classical', f'{q} − {c}', neg(f'Hungarian {d:+.3f} ({lo:.3f} to {hi:.3f}); Long Beach {dv:+.3f} ({lov:.3f} to {hiv:.3f})')])
for x in nzTE:
    if x['p2'] in (0.0, 0.01, 0.05, 0.10):
        S7.append(['Gate-level noise, p₂ = ' + f"{x['p2']:g}", 'VQC-TE, ' + ('exact' if x['shots'] is None else '1024 shots'),
                   f"Ext AUC {x['AUC']:.3f}; Brier {x['Brier']:.3f}; mean |⟨Z₀⟩| {x['mean_abs_Z']:.3f}"])
WS7 = [2700, 2300, 4026]
S7note = ('Ext, external (Hungarian). ΔAUC, paired bootstrap difference in AUC (1000 resamples) with 95% CI; intervals that '
          'include negative values are written with “to” to avoid ambiguity with the minus sign. Training times depend on the hardware. The noise '
          'results for VQC-DR (pre-specified seed) are shown in figure 6F.')


TABLES = {
    'Table 1': ('Classical model performance: leakage-free five-fold cross-validation (internal, Cleveland, n=303) and external validation (Hungarian cohort, n=294; Long Beach cohort, n=200)', TAB1),
    'Table 2': ('Quantum model performance and matched-input classical comparators (development n=303; external Hungarian n=294)', TAB2),
    'Table S1': ('Model specification', S1),
    'Table S2': ('Software environment', S2),
    'Table S3': ('Ablation design', S3),
    'Table S4': ('Baseline characteristics, missing data and unadjusted associations', S4),
    'Table S5': ('Learning curves: external AUC by development training size', S5),
    'Table S6': ('External AUC by sex and age band (Hungarian cohort)', S6),
    'Table S7': ('Sensitivity and secondary analyses', S7),
}
NOTES = {'Table S3': S3note, 'Table S4': S4note, 'Table S5': S5note, 'Table S6': S6note, 'Table S7': S7note}
# cells that depend on the speed of the computer and are therefore not compared with the article
TIMING_CELLS = {('Table 2', r, 11) for r in range(1, 7)} | {('Table S2', 7, 1)} | \
               {('Table S7', r, 2) for r, row in enumerate(S7) if row[0].startswith('200 training epochs')}


def export(outdir='tables'):
    os.makedirs(outdir, exist_ok=True)
    from openpyxl import Workbook
    from openpyxl.styles import Font, Alignment
    wb = Workbook(); wb.remove(wb.active)
    for name, (title, rows) in TABLES.items():
        if rows is None: continue
        fn = name.replace(' ', '_')
        with open(os.path.join(outdir, fn + '.csv'), 'w', newline='', encoding='utf-8-sig') as f:
            w = csv.writer(f); w.writerow([f'{name}. {title}']); w.writerows(rows)
            if name in NOTES: w.writerow([NOTES[name]])
        ws = wb.create_sheet(fn); ws.append([f'{name}. {title}']); ws['A1'].font = Font(bold=True)
        for r in rows: ws.append(r)
        for c in ws[2]: c.font = Font(bold=True)
        if name in NOTES: ws.append([]); ws.append([NOTES[name]])
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(60, max(10, max(len(str(c.value or '')) for c in col[1:]) + 2))
            for c in col: c.alignment = Alignment(wrap_text=True, vertical='top')
    wb.save(os.path.join(outdir, 'All_tables.xlsx'))
    return [n for n, (_, r) in TABLES.items() if r is not None]


if __name__ == '__main__':
    print('written:', export())


## 3. Run the analysis (each script writes `res_*.json`; later steps read earlier outputs)

In [ ]:
import subprocess, sys, time
t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_a_classical.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:])
assert r.returncode == 0, r.stderr[-3000:]
print(f'step A finished in {time.time() - t0:.0f} s')

In [ ]:
import subprocess, sys, time
t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_b_quantum.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:])
assert r.returncode == 0, r.stderr[-3000:]
print(f'step B finished in {time.time() - t0:.0f} s')

In [ ]:
import subprocess, sys, time
t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_d_misc.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:])
assert r.returncode == 0, r.stderr[-3000:]
print(f'step D finished in {time.time() - t0:.0f} s')

In [ ]:
import subprocess, sys, time
t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_e_ablation.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:])
assert r.returncode == 0, r.stderr[-3000:]
print(f'step E finished in {time.time() - t0:.0f} s')

In [ ]:
import subprocess, sys, time
t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_f_noise.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:])
assert r.returncode == 0, r.stderr[-3000:]
print(f'step F finished in {time.time() - t0:.0f} s')

In [ ]:
import subprocess, sys, time
t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_g_paired.py'], capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:])
assert r.returncode == 0, r.stderr[-3000:]
print(f'step G finished in {time.time() - t0:.0f} s')

In [ ]:
import subprocess, sys, time
if RUN_LEARNING_CURVES:
    t0 = time.time(); r = subprocess.run([sys.executable, '-u', 'run_c_learning.py'], capture_output=True, text=True)
    print(r.stdout[-2500:]); assert r.returncode == 0, r.stderr[-3000:]
    print(f'step C finished in {time.time() - t0:.0f} s')
else:
    print('learning curves skipped (RUN_LEARNING_CURVES = False)')

## 4. Figures 1–8 and S1–S3

In [ ]:
import numpy, pandas
assert numpy.__version__ == PINS['numpy'] and pandas.__version__ == PINS['pandas'], (
    'The kernel still has an old NumPy/pandas loaded: Runtime -> Restart session, then Run all.')
import importlib, figs
importlib.reload(figs)
todo = [figs.fig1, figs.fig2, figs.fig3, figs.fig5, figs.fig6, figs.fig7, figs.fig8, figs.figS_loss, figs.figS_cal, figs.figS_dca]
if RUN_LEARNING_CURVES: todo.insert(3, figs.fig4)
for f in todo: f()
print(sorted(os.listdir('figures')))
from IPython.display import Image, display
for f in sorted(os.listdir('figures')):
    print(f); display(Image(filename='figures/' + f, width=650))

### Figure check against the article figures (SHA-256 of the decoded pixels)
The check compares the image pixels, not the PNG file bytes, so differences in PNG compression or metadata between platforms do not matter. A different font installation could still change pixels without changing any value, so this check is informative and does not stop the run.

In [ ]:
import hashlib, json, numpy as np
from PIL import Image as PILImage
FIG_SHA = json.loads('''{"fig1.png": "d35c8a80d3eb152992fadd63379f4c4b5b8d8d65ba625e589ecaae2327c9ea37", "fig2.png": "d5772d967a0c252aac8a1985896f228409b71bca8f642f2816c242b315d1ae4c", "fig3.png": "1a5b90c64de9ba1224a2448c28ef23ab6b21f462df4de79bf8ad930c567ec45f", "fig4.png": "a7f2a33c9af2dec2e9097d34a198533ed9eaf45805d7b8ab18f3135e279854a2", "fig5.png": "80c8d5264fd3bbb7d0e276e0a1da6c4784742dbe9da6c163809ed93b5dab055b", "fig6.png": "0355ca178c879297f61ac1a9e44ce6d3f27083834524dfb5a026b884d81df1c0", "fig7.png": "8369ae35abb0c5ac50428be9ce86ba19fd3f6334f9928491d2b10b37e45fd777", "fig8.png": "a0394ed00f6238ae17330f46efd452df4172818efa1b803324c842c57b2fc7fd", "figS1.png": "57aa596ccba04d024deb4435fad67e1cd2acc2ad98e12065fa9c975d0e32ea8e", "figS2.png": "7d89719583c9e669821953efad770d3f5b867bd35193af9d2f47a62a72e92f77", "figS3.png": "e179016c9ee515df8ac011ce5741d5b48afb021e61dadac873fbddc5d4a641c4"}''')
same = 0
for f, h in FIG_SHA.items():
    p = os.path.join('figures', f)
    if not os.path.exists(p): print(f'{f:10s} not produced (learning curves skipped?)'); continue
    ok = hashlib.sha256(np.asarray(PILImage.open(p).convert('RGB')).tobytes()).hexdigest() == h; same += ok
    print(f'{f:10s}', 'pixel-identical to the article figure' if ok else 'pixels differ from the article figure (check fonts)')
print(f'{same} of {len(FIG_SHA)} figures pixel-identical to the article figures')

## 4b. All tables of the article (tables 1 and 2; appendix tables S1 to S7)
The tables are built from the result files, written to `tables/` as CSV files and one Excel workbook (`All_tables.xlsx`), shown below, and compared cell by cell with the tables printed in the article. Cells that report computing time depend on the computer and are not compared.

In [ ]:
import importlib, tables, pandas as pd, json
from IPython.display import display, Markdown
importlib.reload(tables)
written = tables.export('tables')
for name in written:
    title, rows = tables.TABLES[name]
    display(Markdown(f'**{name}.** {title}'))
    display(pd.DataFrame(rows[1:], columns=rows[0]))
    if name in tables.NOTES: print(tables.NOTES[name])
EXPECTED = json.loads('''{"Table 1": [["Model", "Int AUC (CV), mean (SD)", "Ext AUC (95% CI)", "ACC", "SEN", "SPE", "Brier", "Cal int", "Cal slope", "LB AUC (95% CI)"], ["Logistic regression", "0.912 (0.018)", "0.879 (0.836–0.922)", "0.820", "0.632", "0.926", "0.129", "0.28", "1.15", "0.688 (0.600–0.768)"], ["SVM-RBF", "0.898 (0.025)", "0.886 (0.845–0.927)", "0.813", "0.623", "0.920", "0.128", "0.24", "1.29", "0.707 (0.623–0.780)"], ["Random forest", "0.911 (0.024)", "0.881 (0.837–0.923)", "0.820", "0.613", "0.936", "0.135", "0.23", "1.45", "0.737 (0.655–0.811)"], ["MLP", "0.879 (0.030)", "0.781 (0.727–0.834)", "0.724", "0.575", "0.809", "0.233", "0.60", "0.19", "0.612 (0.517–0.701)"], ["XGBoost", "0.890 (0.018)", "0.873 (0.832–0.916)", "0.810", "0.566", "0.947", "0.146", "0.83", "0.67", "0.720 (0.639–0.799)"], ["LightGBM", "0.888 (0.017)", "0.858 (0.814–0.903)", "0.786", "0.585", "0.899", "0.157", "0.82", "0.45", "0.711 (0.626–0.789)"]], "Table 2": [["Model", "Params", "Train AUC", "Int AUC (CV), mean (SD)", "Ext AUC (95% CI)", "Seed range", "ACC", "SEN", "SPE", "Brier", "Cal slope", "Time (s)"], ["VQC-DR", "16", "0.778", "0.783 (0.084)", "0.616 (0.546–0.685)", "0.616–0.879", "0.554", "0.613", "0.521", "0.242", "1.75", "4.1"], ["VQC-3L", "24", "0.878", "0.857 (0.056)", "0.874 (0.826–0.919)", "0.780–0.874", "0.833", "0.651", "0.936", "0.182", "4.37", "8.8"], ["VQC-TE", "32", "0.892", "0.882 (0.039)", "0.881 (0.838–0.923)", "0.789–0.889", "0.837", "0.698", "0.915", "0.176", "4.06", "8.0"], ["QSVM", "—", "0.938", "0.652 (0.060)", "0.619 (0.555–0.680)", "—", "0.609", "0.377", "0.739", "0.226", "0.83", "<1"], ["LR-PCA4", "5", "0.908", "0.903 (0.028)", "0.884 (0.842–0.926)", "—", "0.813", "0.594", "0.936", "0.130", "1.28", "<1"], ["XGBoost-PCA4", "—", "1.000", "0.859 (0.055)", "0.846 (0.799–0.891)", "—", "0.769", "0.491", "0.926", "0.177", "0.51", "<1"]], "Table S1": [["Model", "Type", "Parameters and hyperparameters", "Training", "Uncertainty"], ["Logistic regression", "Classical, linear", "L2 penalty, C = 1.0, max_iter = 1000; continuous predictors linear", "Five-fold stratified CV on Cleveland; refitted on n = 303", "B = 1000 bootstrap"], ["SVM-RBF", "Classical, kernel", "C = 1.0, RBF kernel, Platt probability calibration", "As above", "As above"], ["Random forest", "Classical, bagging", "500 trees; scikit-learn defaults otherwise", "As above", "As above"], ["MLP", "Classical, neural network", "Hidden layers 32 and 16 (ReLU); L2 0.001; Adam; max_iter = 2000", "As above", "As above"], ["XGBoost", "Classical, boosting", "n_estimators = 200, max_depth = 4, learning_rate = 0.1", "As above", "As above"], ["LightGBM", "Classical, boosting", "n_estimators = 200, learning_rate = 0.1", "As above", "As above"], ["LR-PCA4, XGBoost-PCA4", "Classical, matched input", "As logistic regression and XGBoost, on the four angle-scaled principal components", "As above", "As above"], ["VQC-DR", "Quantum, variational", "4 qubits, L = 2 re-uploading blocks, 16 parameters; angle encoding RY(x)·RZ(x); circular CNOT; Adam η = 0.08; exact parameter-shift gradients", "15 full-batch epochs on n = 303; same CV folds", "As above; five seeds"], ["VQC-3L", "Quantum, variational", "As VQC-DR with L = 3, 24 parameters; Adam η = 0.06", "As above", "As above"], ["VQC-TE", "Quantum, variational", "As VQC-DR with trainable encoding scales (16 encoding, 16 variational parameters; scales initialised U(0.8, 1.2)); Adam η = 0.06; finite differences (ε = 0.10) for encoding parameters", "As above", "As above"], ["QSVM", "Quantum, kernel", "ZZ feature map, 2 repetitions, all qubit pairs; exact fidelity kernel; classical SVM C = 1.0", "Kernel on full training data; same CV folds", "B = 1000 bootstrap"], ["Conformal layer", "Distribution-free wrapper", "Split conformal, LAC score, α = 0.10; XGBoost base learner as above", "Proper training n = 227, calibration n = 76", "Empirical coverage"]], "Table S2": [["Component", "Version", "Role"], ["Python", "3.12.3", "Analysis language"], ["scikit-learn", "1.8.0", "Pipelines, imputation, scaling, PCA, classical models, cross-validation, metrics"], ["XGBoost / LightGBM", "3.4.1 / 4.7.0", "Gradient-boosted ensembles; XGBoost is the conformal base learner"], ["PennyLane", "0.45.0", "Statevector (default.qubit) and density-matrix (default.mixed) simulation"], ["MAPIE", "1.4.1", "Split conformal prediction"], ["NumPy / statsmodels", "2.4.4 / 0.15.0", "Numerics; calibration intercept and slope; unadjusted odds ratios"], ["Simulation cost", "3.5 ms per patient; 9 ms per 303 patients (broadcast)", "Full 15-epoch training 4–9 s per circuit on one CPU core"], ["Random seeds", "Fixed and recorded in the analysis code", "Reproduce every split, sub-sample, initialisation and bootstrap"]], "Table S3": [["Factor", "Levels examined", "Reference level", "Reported in"], ["A1 Depth and encoding architecture", "L = 1; L = 2 (VQC-DR); L = 3 (VQC-3L); L = 2 with trainable encoding (VQC-TE)", "L = 2, fixed encoding", "Figure 6A"], ["A2 Entanglement structure", "Linear; circular; full", "Circular", "Figure 6B"], ["A3 Qubit count", "4; 6; 8 (principal components = qubits)", "4 qubits", "Figure 6C"], ["A4 Encoding strategy", "Angle; IQP; amplitude (13 predictors on 4 qubits)", "Angle", "Figure 6D"], ["A5 Data re-uploading", "Enabled; disabled (encode once, two variational blocks)", "Enabled", "Figure 6E"], ["Noise sensitivity", "Two-qubit error p₂ ∈ {0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10}; exact or 1024 shots", "p₂ = 0", "Figure 6F; table S7"]], "Table S4": [["Characteristic", "Cleveland (n=303)", "Hungarian (n=294)", "Long Beach (n=200)", "Missing, n (Cleveland / Hungarian / Long Beach)", "Unadjusted OR (95% CI), Cleveland"], ["Obstructive CAD, n (%)", "139 (45.9)", "106 (36.1)", "149 (74.5)", "0 / 0 / 0", "—"], ["Age, years, mean (SD)", "54.4 (9.0)", "47.8 (7.8)", "59.4 (7.8)", "0 / 0 / 0", "1.60 (1.26–2.04) per SD"], ["Male sex, n (%)", "206 (68.0)", "213 (72.4)", "194 (97.0)", "0 / 0 / 0", "3.57 (2.10–6.07) per coded unit"], ["Asymptomatic chest pain (cp=4), n (%)", "144 (47.5)", "123 (41.8)", "131 (65.5)", "0 / 0 / 0", "2.92 (2.13–4.01) per coded unit"], ["Resting BP, mm Hg, mean (SD)", "131.7 (17.6)", "132.6 (17.6)", "134.7 (18.4)", "0 / 1 / 57", "1.36 (1.08–1.72) per SD"], ["Cholesterol, mg/dL, mean (SD)", "246.7 (51.8)", "250.8 (67.7)", "239.6 (52.8)", "0 / 23 / 56", "1.19 (0.94–1.50) per SD"], ["Fasting glucose >120 mg/dL, n (%)", "45 (14.9)", "20 (7.0)", "68 (35.2)", "0 / 8 / 7", "1.15 (0.61–2.17) per coded unit"], ["Abnormal resting ECG (restecg ≥1), n (%)", "152 (50.2)", "58 (19.8)", "120 (60.0)", "0 / 1 / 0", "1.41 (1.12–1.78) per coded unit"], ["Maximum heart rate, bpm, mean (SD)", "149.6 (22.9)", "139.1 (23.6)", "122.8 (22.0)", "0 / 1 / 53", "0.37 (0.28–0.50) per SD"], ["Exercise-induced angina, n (%)", "99 (32.7)", "89 (30.4)", "95 (64.6)", "0 / 1 / 53", "7.40 (4.25–12.86) per coded unit"], ["ST depression, mm, mean (SD)", "1.0 (1.2)", "0.6 (0.9)", "1.3 (1.1)", "0 / 0 / 56", "2.91 (2.13–3.98) per SD"], ["Flat or downsloping ST (slope ≥2), n (%)", "161 (53.1)", "92 (88.5)", "82 (83.7)", "0 / 190 / 102", "3.32 (2.19–5.04) per coded unit"], ["≥1 vessel on fluoroscopy (ca ≥1), n (%)", "123 (41.1)", "0 (0.0)", "0 (0.0)", "4 / 291 / 198", "3.46 (2.44–4.89) per coded unit"], ["Reversible or fixed defect (thal 6 or 7), n (%)", "135 (44.9)", "21 (75.0)", "30 (88.2)", "2 / 266 / 166", "1.83 (1.59–2.10) per coded unit"]], "Table S5": [["Model", "n=30", "n=60", "n=120", "n=240", "n=303"], ["Logistic regression", "0.819 (0.026)", "0.828 (0.020)", "0.867 (0.009)", "0.874 (0.006)", "0.879"], ["SVM-RBF", "0.852 (0.023)", "0.852 (0.016)", "0.871 (0.018)", "0.879 (0.007)", "0.886"], ["Random forest", "0.822 (0.036)", "0.865 (0.012)", "0.875 (0.021)", "0.878 (0.006)", "0.881"], ["MLP", "0.795 (0.026)", "0.785 (0.051)", "0.813 (0.029)", "0.780 (0.021)", "0.781"], ["XGBoost", "0.773 (0.026)", "0.821 (0.032)", "0.834 (0.038)", "0.840 (0.021)", "0.873"], ["LightGBM", "not estimable", "0.804 (0.035)", "0.824 (0.042)", "0.830 (0.025)", "0.858"], ["LR-PCA4", "0.862 (0.020)", "0.864 (0.031)", "0.880 (0.009)", "0.884 (0.004)", "0.884"], ["VQC-DR", "0.720 (0.133)", "0.792 (0.076)", "0.764 (0.123)", "0.835 (0.050)", "0.616"], ["VQC-3L", "0.694 (0.123)", "0.790 (0.052)", "0.811 (0.042)", "0.800 (0.054)", "0.874"], ["VQC-TE", "0.791 (0.069)", "0.867 (0.015)", "0.857 (0.027)", "0.862 (0.020)", "0.881"]], "Table S6": [["Model", "Women (n=81; 12 events)", "Men (n=213; 94 events)", "Age <55 years (n=236; 80 events)", "Age ≥55 years (n=58; 26 events)"], ["Logistic regression", "0.841 (0.669–0.972)", "0.864 (0.811–0.913)", "0.888 (0.837–0.936)", "0.832 (0.716–0.926)"], ["SVM-RBF", "0.838 (0.667–0.969)", "0.879 (0.830–0.922)", "0.897 (0.850–0.941)", "0.852 (0.737–0.939)"], ["Random forest", "0.809 (0.619–0.961)", "0.877 (0.828–0.923)", "0.894 (0.844–0.938)", "0.821 (0.702–0.924)"], ["MLP", "0.814 (0.667–0.945)", "0.723 (0.654–0.790)", "0.791 (0.729–0.845)", "0.719 (0.577–0.843)"], ["XGBoost", "0.854 (0.727–0.958)", "0.860 (0.810–0.906)", "0.891 (0.843–0.932)", "0.785 (0.662–0.898)"], ["LightGBM", "0.849 (0.687–0.964)", "0.838 (0.784–0.890)", "0.874 (0.822–0.918)", "0.760 (0.638–0.887)"], ["VQC-DR", "0.729 (0.597–0.857)", "0.539 (0.462–0.619)", "0.603 (0.530–0.680)", "0.674 (0.535–0.801)"], ["VQC-3L", "0.762 (0.551–0.949)", "0.889 (0.843–0.934)", "0.875 (0.814–0.929)", "0.851 (0.749–0.934)"], ["VQC-TE", "0.868 (0.745–0.966)", "0.876 (0.829–0.920)", "0.890 (0.835–0.935)", "0.871 (0.764–0.952)"], ["QSVM", "0.607 (0.436–0.769)", "0.641 (0.558–0.713)", "0.602 (0.526–0.676)", "0.641 (0.508–0.779)"]], "Table S7": [["Analysis", "Model", "Result"], ["Without slope, ca and thal (10 predictors)", "Logistic regression", "Ext AUC 0.880 (0.837–0.921) vs 0.879 with 13"], ["Without slope, ca and thal (10 predictors)", "SVM-RBF", "Ext AUC 0.877 (0.838–0.916) vs 0.886 with 13"], ["Without slope, ca and thal (10 predictors)", "Random forest", "Ext AUC 0.881 (0.839–0.922) vs 0.881 with 13"], ["Without slope, ca and thal (10 predictors)", "MLP", "Ext AUC 0.813 (0.760–0.863) vs 0.781 with 13"], ["Without slope, ca and thal (10 predictors)", "XGBoost", "Ext AUC 0.853 (0.809–0.899) vs 0.873 with 13"], ["Without slope, ca and thal (10 predictors)", "LightGBM", "Ext AUC 0.840 (0.792–0.887) vs 0.858 with 13"], ["Five initialisation seeds (15 epochs)", "VQC-DR", "Ext AUC 0.616, 0.874, 0.879, 0.652, 0.871 (seeds 42, 1, 2, 3, 4)"], ["Five initialisation seeds (15 epochs)", "VQC-3L", "Ext AUC 0.874, 0.859, 0.871, 0.824, 0.780 (seeds 42, 1, 2, 3, 4)"], ["Five initialisation seeds (15 epochs)", "VQC-TE", "Ext AUC 0.881, 0.877, 0.889, 0.789, 0.831 (seeds 42, 1, 2, 3, 4)"], ["200 training epochs (seed 42)", "VQC-DR", "Ext AUC 0.885 (0.843–0.924); Brier 0.175; 55 s"], ["200 training epochs (seed 42)", "VQC-3L", "Ext AUC 0.872 (0.828–0.914); Brier 0.174; 114 s"], ["200 training epochs (seed 42)", "VQC-TE", "Ext AUC 0.861 (0.814–0.908); Brier 0.167; 108 s"], ["Paired ΔAUC, quantum − classical", "VQC-TE − Logistic regression", "Hungarian +0.003 (−0.019 to 0.025); Long Beach +0.050 (0.004 to 0.093)"], ["Paired ΔAUC, quantum − classical", "VQC-TE − SVM-RBF", "Hungarian −0.005 (−0.027 to 0.017); Long Beach +0.032 (−0.003 to 0.065)"], ["Paired ΔAUC, quantum − classical", "VQC-TE − Random forest", "Hungarian 0.000 (−0.022 to 0.021); Long Beach +0.002 (−0.044 to 0.045)"], ["Paired ΔAUC, quantum − classical", "VQC-TE − XGBoost", "Hungarian +0.008 (−0.021 to 0.039); Long Beach +0.018 (−0.048 to 0.084)"], ["Paired ΔAUC, quantum − classical", "VQC-TE − LightGBM", "Hungarian +0.023 (−0.008 to 0.054); Long Beach +0.028 (−0.034 to 0.090)"], ["Paired ΔAUC, quantum − classical", "VQC-TE − LR-PCA4", "Hungarian −0.003 (−0.018 to 0.014); Long Beach +0.006 (−0.013 to 0.025)"], ["Paired ΔAUC, quantum − classical", "VQC-3L − Logistic regression", "Hungarian −0.005 (−0.037 to 0.029); Long Beach +0.011 (−0.047 to 0.071)"], ["Paired ΔAUC, quantum − classical", "VQC-3L − SVM-RBF", "Hungarian −0.012 (−0.042 to 0.020); Long Beach −0.008 (−0.052 to 0.041)"], ["Paired ΔAUC, quantum − classical", "VQC-3L − Random forest", "Hungarian −0.007 (−0.037 to 0.023); Long Beach −0.038 (−0.099 to 0.026)"], ["Paired ΔAUC, quantum − classical", "VQC-3L − XGBoost", "Hungarian +0.001 (−0.037 to 0.039); Long Beach −0.021 (−0.100 to 0.054)"], ["Paired ΔAUC, quantum − classical", "VQC-3L − LightGBM", "Hungarian +0.016 (−0.022 to 0.054); Long Beach −0.011 (−0.085 to 0.061)"], ["Paired ΔAUC, quantum − classical", "VQC-3L − LR-PCA4", "Hungarian −0.010 (−0.037 to 0.019); Long Beach −0.034 (−0.084 to 0.021)"], ["Gate-level noise, p₂ = 0", "VQC-TE, exact", "Ext AUC 0.881; Brier 0.176; mean |⟨Z₀⟩| 0.481"], ["Gate-level noise, p₂ = 0", "VQC-TE, 1024 shots", "Ext AUC 0.882; Brier 0.176; mean |⟨Z₀⟩| 0.481"], ["Gate-level noise, p₂ = 0.01", "VQC-TE, exact", "Ext AUC 0.881; Brier 0.188; mean |⟨Z₀⟩| 0.382"], ["Gate-level noise, p₂ = 0.01", "VQC-TE, 1024 shots", "Ext AUC 0.879; Brier 0.188; mean |⟨Z₀⟩| 0.383"], ["Gate-level noise, p₂ = 0.05", "VQC-TE, exact", "Ext AUC 0.881; Brier 0.224; mean |⟨Z₀⟩| 0.144"], ["Gate-level noise, p₂ = 0.05", "VQC-TE, 1024 shots", "Ext AUC 0.869; Brier 0.224; mean |⟨Z₀⟩| 0.145"], ["Gate-level noise, p₂ = 0.1", "VQC-TE, exact", "Ext AUC 0.881; Brier 0.244; mean |⟨Z₀⟩| 0.036"], ["Gate-level noise, p₂ = 0.1", "VQC-TE, 1024 shots", "Ext AUC 0.831; Brier 0.244; mean |⟨Z₀⟩| 0.047"]]}''')
norm = lambda x: ' '.join(str(x).split())
n_cmp = n_diff = n_time = 0
for name in written:
    gen = tables.TABLES[name][1]; exp = EXPECTED[name]
    assert len(gen) == len(exp) and all(len(a) == len(b) for a, b in zip(gen, exp)), f'{name}: table shape differs'
    for i, (a, b) in enumerate(zip(gen, exp)):
        for j, (x, y) in enumerate(zip(a, b)):
            if (name, i, j) in tables.TIMING_CELLS: n_time += 1; continue
            n_cmp += 1
            if norm(x) != norm(y): n_diff += 1; print('DIFFERS', name, 'row', i, 'col', j, repr(norm(x)), 'article:', repr(y))
print(f'{len(written)} tables, {n_cmp} cells compared with the article: {n_cmp - n_diff} identical, {n_diff} differing '
      f'({n_time} computing-time cells not compared)')
assert n_diff == 0
print('files:', sorted(os.listdir('tables')))

## 5. Verification: every key value reported in the article must be reproduced

In [ ]:
import json, numpy as np
REF = json.loads('''{"a.cohorts.cleveland.n": 303, "a.cohorts.cleveland.events": 139, "a.cohorts.hungarian.n": 294, "a.cohorts.hungarian.events": 106, "a.cohorts.va.n": 200, "a.cohorts.va.events": 149, "a.cohorts.switzerland.n": 123, "a.cohorts.switzerland.events": 115, "a.audit.kaggle_label_inverted": 303, "a.audit.unique_source_records": 191, "a.audit.synthetic_n": 294, "a.Logistic regression.int_auc": 0.9116306918390251, "a.Logistic regression.ext.AUC": 0.8786130068245686, "a.Logistic regression.ext_ci.lo": 0.8357575834097044, "a.Logistic regression.ext_ci.hi": 0.9217239363594775, "a.Logistic regression.ext.Brier": 0.1289196721101675, "a.Logistic regression.ext.CalSlope": 1.1481777688762804, "a.Logistic regression.va.AUC": 0.6881168574812475, "a.Logistic regression.ext10.AUC": 0.879817342432758, "a.SVM-RBF.int_auc": 0.8975075657367324, "a.SVM-RBF.ext.AUC": 0.8864411882778, "a.SVM-RBF.ext_ci.lo": 0.8450622403174005, "a.SVM-RBF.ext_ci.hi": 0.9272820759880224, "a.SVM-RBF.ext.Brier": 0.1275999444602198, "a.SVM-RBF.ext.CalSlope": 1.2873361233969056, "a.SVM-RBF.va.AUC": 0.7068035267798395, "a.SVM-RBF.ext10.AUC": 0.8771075873143316, "a.Random forest.int_auc": 0.9109902346881513, "a.Random forest.ext.AUC": 0.8810216780409472, "a.Random forest.ext_ci.lo": 0.8371835206107747, "a.Random forest.ext_ci.hi": 0.9231623836492138, "a.Random forest.ext.Brier": 0.13506877551020405, "a.Random forest.ext.CalSlope": 1.4543920484455137, "a.Random forest.va.AUC": 0.7367416765363864, "a.Random forest.ext10.AUC": 0.8813227619429949, "a.MLP.int_auc": 0.8790188592271925, "a.MLP.ext.AUC": 0.7806101967081495, "a.MLP.ext_ci.lo": 0.7272981844104605, "a.MLP.ext_ci.hi": 0.834110057319349, "a.MLP.ext.Brier": 0.23297032211811705, "a.MLP.ext.CalSlope": 0.18991100059963903, "a.MLP.va.AUC": 0.611527832609554, "a.MLP.ext10.AUC": 0.8134283420313128, "a.XGBoost.int_auc": 0.8898133116883118, "a.XGBoost.ext.AUC": 0.8733942191890808, "a.XGBoost.ext_ci.lo": 0.8318795669717818, "a.XGBoost.ext_ci.hi": 0.9156832660402472, "a.XGBoost.ext.Brier": 0.1455434262752533, "a.XGBoost.ext.CalSlope": 0.6738409190314908, "a.XGBoost.va.AUC": 0.7203579418344519, "a.XGBoost.ext10.AUC": 0.8528201525491771, "a.LightGBM.int_auc": 0.8880914802789803, "a.LightGBM.ext.AUC": 0.8583902047370535, "a.LightGBM.ext_ci.lo": 0.8143651835864364, "a.LightGBM.ext_ci.hi": 0.9025485675039246, "a.LightGBM.ext.Brier": 0.1571950682651267, "a.LightGBM.ext.CalSlope": 0.44655682685383313, "a.LightGBM.va.AUC": 0.7106198183971576, "a.LightGBM.ext10.AUC": 0.8403251706142112, "b.VQC-DR.int_auc": 0.7829149631232964, "b.VQC-DR.ext.AUC": 0.6163689281413087, "b.VQC-DR.ext.Brier": 0.2424527981393694, "b.VQC-DR.va.AUC": 0.4813791288327411, "b.VQC-DR.ext_ci.lo": 0.5464480230857919, "b.VQC-DR.ext_ci.hi": 0.6846374019324837, "b.VQC-DR.ext.CalSlope": 1.7460067225055498, "b.VQC-DR.seed42": 0.6163689281413087, "b.VQC-DR.seed1": 0.8738458450421518, "b.VQC-DR.seed2": 0.8793657165796869, "b.VQC-DR.seed3": 0.6515455640305099, "b.VQC-DR.seed4": 0.8711862705740665, "b.VQC-DR.ext200.AUC": 0.8845343235648334, "b.VQC-3L.int_auc": 0.8568810626102292, "b.VQC-3L.ext.AUC": 0.8740465676435167, "b.VQC-3L.ext.Brier": 0.18208068409109907, "b.VQC-3L.va.AUC": 0.6991709435452034, "b.VQC-3L.ext_ci.lo": 0.8262168855293087, "b.VQC-3L.ext_ci.hi": 0.9185990431456912, "b.VQC-3L.ext.CalSlope": 4.372449061311028, "b.VQC-3L.seed42": 0.8740465676435167, "b.VQC-3L.seed1": 0.8587916499397833, "b.VQC-3L.seed2": 0.8711862705740667, "b.VQC-3L.seed3": 0.8242673625050181, "b.VQC-3L.seed4": 0.7804094741067844, "b.VQC-3L.ext200.AUC": 0.8720895222802089, "b.VQC-TE.int_auc": 0.8817773468815135, "b.VQC-TE.ext.AUC": 0.8811722199919712, "b.VQC-TE.ext.Brier": 0.17644890762533225, "b.VQC-TE.va.AUC": 0.7385182260823794, "b.VQC-TE.ext_ci.lo": 0.8381469585150013, "b.VQC-TE.ext_ci.hi": 0.9225419860444507, "b.VQC-TE.ext.CalSlope": 4.060375943960503, "b.VQC-TE.seed42": 0.8811722199919712, "b.VQC-TE.seed1": 0.8768566840626254, "b.VQC-TE.seed2": 0.8887494981934966, "b.VQC-TE.seed3": 0.7891409072661582, "b.VQC-TE.seed4": 0.83129265355279, "b.VQC-TE.ext200.AUC": 0.8607486953030911, "b.QSVM.int_auc": 0.6523897206188873, "b.QSVM.ext.AUC": 0.6186521477318346, "b.QSVM.ext.Brier": 0.22571334030072043, "b.QSVM.va.AUC": 0.5271746282405579, "b.QSVM.ext_ci.lo": 0.5551489526220692, "b.QSVM.ext_ci.hi": 0.6802874918430739, "b.QSVM.ext.CalSlope": 0.8335406946510457, "b.LR-PCA4.int_auc": 0.9034391534391535, "b.LR-PCA4.ext.AUC": 0.8841830590124448, "b.LR-PCA4.ext.Brier": 0.12968027789664033, "b.LR-PCA4.va.AUC": 0.7329911830504013, "b.LR-PCA4.ext_ci.lo": 0.842481475037381, "b.LR-PCA4.ext_ci.hi": 0.9256693575949024, "b.LR-PCA4.ext.CalSlope": 1.2803523275348228, "b.XGBoost-PCA4.int_auc": 0.8593379228795897, "b.XGBoost-PCA4.ext.AUC": 0.8461963067041349, "b.XGBoost-PCA4.ext.Brier": 0.17660152912139893, "b.XGBoost-PCA4.va.AUC": 0.7056849585471773, "b.XGBoost-PCA4.ext_ci.lo": 0.7987472510868028, "b.XGBoost-PCA4.ext_ci.hi": 0.890917150617175, "b.XGBoost-PCA4.ext.CalSlope": 0.5083939042525684, "b.QSVM.kta": 0.0424934678718183, "d.conformal.internal_cv.coverage": 0.8978688524590164, "d.conformal.internal_cv.size": 1.2608743169398906, "d.conformal.internal_cv.cov_y1": 0.87010582010582, "d.conformal.internal_cv.cov_y0": 0.9212121212121213, "d.conformal.external.coverage": 0.9285714285714286, "d.conformal.external.size": 1.5068027210884354, "d.conformal.external.cov_y1": 0.8018867924528302, "d.conformal.external.cov_y0": 1.0, "d.conformal.va.coverage": 0.895, "d.conformal.va.size": 1.745, "d.conformal.va.cov_y1": 0.8791946308724832, "d.conformal.va.cov_y0": 0.9411764705882353, "d.gradvar.2": 0.06553560337313152, "d.gradvar.4": 0.020533607254479277, "d.gradvar.6": 0.005688208855858817, "d.gradvar.8": 0.001595211279900021, "d.gradvar.slope": -0.2699233493931656, "f.VQC-TE.p0.0.exact.AUC": 0.8811722199919712, "f.VQC-TE.p0.0.shots.AUC": 0.8821005620232838, "f.VQC-TE.p0.001.exact.AUC": 0.8811220393416299, "f.VQC-TE.p0.001.shots.AUC": 0.8765806904857487, "f.VQC-TE.p0.005.exact.AUC": 0.8811220393416298, "f.VQC-TE.p0.005.shots.AUC": 0.8817743877960659, "f.VQC-TE.p0.01.exact.AUC": 0.8810216780409474, "f.VQC-TE.p0.01.shots.AUC": 0.8787886391007627, "f.VQC-TE.p0.02.exact.AUC": 0.8810216780409474, "f.VQC-TE.p0.02.shots.AUC": 0.878387193898033, "f.VQC-TE.p0.05.exact.AUC": 0.8811722199919712, "f.VQC-TE.p0.05.shots.AUC": 0.8687775993576876, "f.VQC-TE.p0.1.exact.AUC": 0.8812725812926536, "f.VQC-TE.p0.1.shots.AUC": 0.8312173825772783, "e.A1 L=1.mean": 0.4930349257326375, "e.A1 L=2 (VQC-DR).mean": 0.7784624648735448, "e.A1 L=3 (VQC-3L).mean": 0.8417402649538339, "e.A1 L=2 trainable (VQC-TE).mean": 0.8534423926134082, "e.A2 linear.mean": 0.8345945403452429, "e.A2 circular.mean": 0.7784624648735448, "e.A2 full.mean": 0.8493978321959051, "e.A3 4 qubits.mean": 0.7784624648735448, "e.A3 6 qubits.mean": 0.791619831393015, "e.A3 8 qubits.mean": 0.6736651947009233, "e.A4 angle.mean": 0.7784624648735448, "e.A4 IQP.mean": 0.7542151746286632, "e.A4 amplitude.mean": 0.6013448414291449, "e.A5 re-uploading on.mean": 0.7784624648735448, "e.A5 re-uploading off.mean": 0.8384985949417905, "c.Logistic regression.n30": 0.8190485748695304, "c.Logistic regression.n60": 0.8284825371336814, "c.Logistic regression.n120": 0.866830590124448, "c.Logistic regression.n240": 0.8738659173022881, "c.Logistic regression.n303": 0.8786130068245686, "c.SVM-RBF.n30": 0.8524939783219591, "c.SVM-RBF.n60": 0.8517362505018067, "c.SVM-RBF.n120": 0.8708851866720193, "c.SVM-RBF.n240": 0.8789993978321959, "c.SVM-RBF.n303": 0.8864411882778, "c.Random forest.n30": 0.8218787635487754, "c.Random forest.n60": 0.8649688879967885, "c.Random forest.n120": 0.8751154154957848, "c.Random forest.n240": 0.8781212364512244, "c.Random forest.n303": 0.8810216780409472, "c.MLP.n30": 0.7951926936973103, "c.MLP.n60": 0.7849458048976314, "c.MLP.n120": 0.8134082697711762, "c.MLP.n240": 0.7804395824969891, "c.MLP.n303": 0.7806101967081495, "c.XGBoost.n30": 0.7731934965877157, "c.XGBoost.n60": 0.8208600963468486, "c.XGBoost.n120": 0.8342432757928542, "c.XGBoost.n240": 0.839582496989161, "c.XGBoost.n303": 0.8733942191890808, "c.LightGBM.n30": 0.5, "c.LightGBM.n60": 0.8039843436370934, "c.LightGBM.n120": 0.824468085106383, "c.LightGBM.n240": 0.830068245684464, "c.LightGBM.n303": 0.8583902047370535, "c.LR-PCA4.n30": 0.8618225612203932, "c.LR-PCA4.n60": 0.8637595343235649, "c.LR-PCA4.n120": 0.8799377759935769, "c.LR-PCA4.n240": 0.8839421918908069, "c.LR-PCA4.n303": 0.8841830590124448, "c.VQC-DR.n30": 0.7201425130469691, "c.VQC-DR.n60": 0.7916399036531513, "c.VQC-DR.n120": 0.7641007627458851, "c.VQC-DR.n240": 0.83512645523886, "c.VQC-DR.n303": 0.6163689281413087, "c.VQC-3L.n30": 0.6940385387394621, "c.VQC-3L.n60": 0.7902549177037335, "c.VQC-3L.n120": 0.8114010437575271, "c.VQC-3L.n240": 0.799959855479727, "c.VQC-3L.n303": 0.8740465676435167, "c.VQC-TE.n30": 0.7911682055399437, "c.VQC-TE.n60": 0.8672822159775191, "c.VQC-TE.n120": 0.8565937374548375, "c.VQC-TE.n240": 0.8619630670413487, "c.VQC-TE.n303": 0.8811722199919712, "g.VQC-TE vs Logistic regression.diff": 0.0025592131674025964, "g.VQC-TE vs Logistic regression.lo": -0.01913510596546309, "g.VQC-TE vs Logistic regression.hi": 0.025278605115335133, "g.VQC-TE vs Logistic regression [VA].diff": 0.05040136860113187, "g.VQC-TE vs Logistic regression [VA].lo": 0.004273769154649161, "g.VQC-TE vs Logistic regression [VA].hi": 0.09263734043747837, "g.VQC-TE vs LR-PCA4.diff": -0.00301083902047361, "g.VQC-TE vs LR-PCA4.lo": -0.018354944276924332, "g.VQC-TE vs LR-PCA4.hi": 0.014153979258518974, "g.VQC-3L vs Logistic regression.diff": -0.004566439181051818, "g.VQC-3L vs Logistic regression.lo": -0.03728560212835762, "g.VQC-3L vs Logistic regression.hi": 0.029168408001610576, "d.subgroups.Women.VQC-TE": 0.8683574879227053}''')
def extract(J):
    A, B, D, E, F, G = J['a'], J['b'], J['d'], J['e'], J['f'], J['g']; out = {}
    for site in ['cleveland', 'hungarian', 'va', 'switzerland']:
        out[f'a.cohorts.{site}.n'] = A['cohorts'][site]['n']; out[f'a.cohorts.{site}.events'] = A['cohorts'][site]['events']
    for k in ['kaggle_label_inverted', 'unique_source_records', 'synthetic_n']: out[f'a.audit.{k}'] = A['first_submission_audit'][k]
    for m, r in A['classical'].items():
        out[f'a.{m}.int_auc'] = r['int_auc']; out[f'a.{m}.ext.AUC'] = r['ext']['AUC']; out[f'a.{m}.ext_ci.lo'] = r['ext_ci'][0]
        out[f'a.{m}.ext_ci.hi'] = r['ext_ci'][1]; out[f'a.{m}.ext.Brier'] = r['ext']['Brier']; out[f'a.{m}.ext.CalSlope'] = r['ext']['CalSlope']
        out[f'a.{m}.va.AUC'] = r['va']['AUC']; out[f'a.{m}.ext10.AUC'] = r['ext10']['AUC']
    for m in ['VQC-DR', 'VQC-3L', 'VQC-TE', 'QSVM', 'LR-PCA4', 'XGBoost-PCA4']:
        r = B[m]; out[f'b.{m}.int_auc'] = r['int_auc']; out[f'b.{m}.ext.AUC'] = r['ext']['AUC']; out[f'b.{m}.ext.Brier'] = r['ext']['Brier']
        out[f'b.{m}.va.AUC'] = r['va']['AUC']; out[f'b.{m}.ext_ci.lo'] = r['ext_ci'][0]; out[f'b.{m}.ext_ci.hi'] = r['ext_ci'][1]
        out[f'b.{m}.ext.CalSlope'] = r['ext']['CalSlope']
        if 'seed_ext_auc' in r:
            for i, s in enumerate([42, 1, 2, 3, 4]): out[f'b.{m}.seed{s}'] = r['seed_ext_auc'][i]
            out[f'b.{m}.ext200.AUC'] = r['ext200']['AUC']
    out['b.QSVM.kta'] = B['QSVM']['kta']
    for k in ['internal_cv', 'external', 'va']:
        for q in ['coverage', 'size', 'cov_y1', 'cov_y0']: out[f'd.conformal.{k}.{q}'] = D['conformal'][k][q]
    for n in ['2', '4', '6', '8']: out[f'd.gradvar.{n}'] = D['gradvar']['per_qubit'][n]['mean_var_all']
    out['d.gradvar.slope'] = D['gradvar']['slope_log10_all']
    for x in F['VQC-TE']: out[f"f.VQC-TE.p{x['p2']}.{'exact' if x['shots'] is None else 'shots'}.AUC"] = x['AUC']
    for k, v in E.items(): out[f'e.{k}.mean'] = v['mean']
    if 'c' in J:
        for m in J['c']:
            if m.endswith('_time'): continue
            for n in ['30', '60', '120', '240', '303']: out[f'c.{m}.n{n}'] = float(np.mean(J['c'][m][n]))
    for k in ['VQC-TE vs Logistic regression', 'VQC-TE vs Logistic regression [VA]', 'VQC-TE vs LR-PCA4', 'VQC-3L vs Logistic regression']:
        out[f'g.{k}.diff'] = G[k][0]; out[f'g.{k}.lo'] = G[k][1]; out[f'g.{k}.hi'] = G[k][2]
    out['d.subgroups.Women.VQC-TE'] = D['subgroups']['Women']['VQC-TE']['AUC']
    return out
J = {k: json.load(open(f'res_{k}.json')) for k in 'abdefg'}
if os.path.exists('res_c.json') and RUN_LEARNING_CURVES: J['c'] = json.load(open('res_c.json'))
NEW = extract(J)
exact = near = fail = 0; rows = []
for k, v in REF.items():
    if k not in NEW: continue
    d = abs(NEW[k] - v)
    if d < 1e-9: exact += 1
    elif d <= 0.0005: near += 1; rows.append(('NEAR', k, v, NEW[k]))
    else: fail += 1; rows.append(('FAIL', k, v, NEW[k]))
checked = sum(k in NEW for k in REF)
print(f'{checked} reported values checked: {exact} identical, {near} within 0.0005 (rounding-invisible), {fail} differing')
for r in rows: print('  ', r)
assert fail == 0, 'Some values differ from the article beyond rounding: see list above.'

## 6. Independent audit (re-implementation without the analysis code's metric, simulator or conformal libraries)

In [ ]:
%%writefile a1_data_metrics.py
"""Independent audit 1: raw data counts, label audit, and metric recomputation
from saved predictions with independent implementations (no sklearn metrics)."""
import csv, json, pickle, numpy as np
D='data'; A=json.load(open('res_a.json')); B=json.load(open('res_b.json'))
PA=pickle.load(open('pred_a.pkl','rb')); PB=pickle.load(open('pred_b.pkl','rb'))
ok=lambda c,m: print(('PASS ' if c else 'FAIL ')+m)
def raw(site):
    rows=[r for r in csv.reader(open(f'{D}/processed.{site}.data')) if r]
    return rows
# --- counts from raw text
for site,key in [('cleveland','cleveland'),('hungarian','hungarian'),('va','va'),('switzerland','switzerland')]:
    rows=raw(site); n=len(rows); ev=sum(float(r[13])>0 for r in rows)
    age=np.mean([float(r[0]) for r in rows]); male=np.mean([float(r[1]) for r in rows])
    miss_ca=sum(r[11]=='?' for r in rows); miss_thal=sum(r[12]=='?' for r in rows)
    chol0=sum(r[4]!='?' and float(r[4])==0 for r in rows); cholq=sum(r[4]=='?' for r in rows)
    c=A['cohorts'][key]
    ok(n==c['n'] and ev==c['events'], f'{site}: n={n} events={ev} (json {c["n"]},{c["events"]})')
    ok(abs(age-c['age']['mean'])<1e-9, f'{site}: mean age {age:.2f}; male {male:.3f}')
    ok(miss_ca==c['ca']['miss'] and miss_thal==c['thal']['miss'] and chol0+cholq==c['chol']['miss'],
       f'{site}: missing ca={miss_ca} thal={miss_thal} chol(?+0)={cholq}+{chol0}')
# --- label inversion audit, independent: match Kaggle rows to UCI rows on 6 continuous/binary fields
k=list(csv.reader(open(f'{D}/kaggle_heart.csv',encoding='utf-8-sig')))[1:]
cl=raw('cleveland')
idx={}
for r in cl:
    idx.setdefault((float(r[0]),float(r[1]),float(r[3]),float(r[4]),float(r[7]),float(r[9])),[]).append(int(float(r[13])>0))
match=inv=amb=0
for r in k:
    key=(float(r[0]),float(r[1]),float(r[3]),float(r[4]),float(r[7]),float(r[9]))
    if key in idx:
        labs=set(idx[key]); match+=1
        if len(labs)>1: amb+=1
        elif int(r[13])!=list(labs)[0]: inv+=1
ok(match==303 and inv==303-amb, f'Kaggle rows matched {match}/303, label inverted {inv}, ambiguous keys {amb}')
ok(sum(int(r[13]) for r in k)==165, 'Kaggle target=1 count 165 (manuscript v1 "165 with disease")')
# --- independent metrics
def auc(y,p):  # Mann-Whitney with ties
    y=np.asarray(y); p=np.asarray(p); pos=p[y==1]; neg=p[y==0]
    gt=(pos[:,None]>neg[None,:]).sum(); eq=(pos[:,None]==neg[None,:]).sum()
    return (gt+0.5*eq)/(len(pos)*len(neg))
ye=PB['y_ext']; yv=PB['y_va']; y=PB['y_int']
ok(ye.sum()==106 and len(ye)==294 and yv.sum()==149 and y.sum()==139, 'saved outcome vectors match cohort counts')
from scipy.optimize import minimize
def calib(y,p):
    p=np.clip(p,1e-6,1-1e-6); lp=np.log(p/(1-p))
    nll=lambda b: -np.sum(y*(b[0]+b[1]*lp)-np.log1p(np.exp(b[0]+b[1]*lp)))
    slope=minimize(nll,[0,1],method='BFGS').x[1]
    nll2=lambda a: -np.sum(y*(a[0]+lp)-np.log1p(np.exp(a[0]+lp)))
    inter=minimize(nll2,[0],method='BFGS').x[0]
    return inter,slope
maxdiff=0
for name in list(A['classical'])+['VQC-DR','VQC-3L','VQC-TE','QSVM','LR-PCA4','XGBoost-PCA4']:
    r=A['classical'][name] if name in A['classical'] else B[name]
    P=PA[name] if name in PA else PB[name]
    pe=P['ext']; a=auc(ye,pe); yp=(pe>=0.5).astype(int)
    sen=((yp==1)&(ye==1)).sum()/ye.sum(); spe=((yp==0)&(ye==0)).sum()/(ye==0).sum(); br=np.mean((pe-ye)**2)
    ci,cs=calib(ye,pe); va=auc(yv,P['va'])
    d=max(abs(a-r['ext']['AUC']),abs(sen-r['ext']['SEN']),abs(spe-r['ext']['SPE']),abs(br-r['ext']['Brier']),abs(va-r['va']['AUC']),abs(ci-r['ext']['CalInt']),abs(cs-r['ext']['CalSlope']))
    maxdiff=max(maxdiff,d)
    # CV mean from folds requires fold AUC: recompute from oof per fold
    fa=[auc(y[te],P['oof'][te]) for tr,te in PB['folds']]
    ok(abs(np.mean(fa)-r['int_auc'])<1e-9 and d<1e-3, f'{name:20s} AUC {a:.4f} SEN {sen:.3f} SPE {spe:.3f} Brier {br:.4f} cal {ci:.3f}/{cs:.3f} VA {va:.3f} CV {np.mean(fa):.4f} (maxdiff {d:.1e})')
# folds: stratified, disjoint, cover all
f=PB['folds']; allte=np.concatenate([te for _,te in f])
ok(sorted(allte)==list(range(303)) and all(len(set(tr)&set(te))==0 for tr,te in f), 'CV folds disjoint and cover all 303')
ok(all(abs(y[te].mean()-y.mean())<0.02 for _,te in f), 'CV folds stratified: '+str([round(y[te].mean(),3) for _,te in f]))
# bootstrap CI stability with a different seed
rng=np.random.RandomState(7)
for name in ['Logistic regression','VQC-TE']:
    P=PA[name] if name in PA else PB[name]; pe=P['ext']; v=[]
    while len(v)<2000:
        i=rng.randint(0,294,294)
        if 0<ye[i].sum()<294: v.append(auc(ye[i],pe[i]))
    lo,hi=np.percentile(v,[2.5,97.5]); r=A['classical'][name]['ext_ci'] if name in A['classical'] else B[name]['ext_ci']
    ok(abs(lo-r[0])<0.006 and abs(hi-r[1])<0.006, f'{name} bootstrap CI reseeded {lo:.3f}-{hi:.3f} vs reported {r[0]:.3f}-{r[1]:.3f}')
# min/max predicted probability of quantum models (sigmoid bound claim)
for q in ['VQC-DR','VQC-3L','VQC-TE']:
    pe=PB[q]['ext']; print(f'INFO {q} external prediction range {pe.min():.3f}-{pe.max():.3f}')
ok(1/(1+np.exp(1))<0.27 and 1/(1+np.exp(-1))>0.73-0.001, 'sigmoid([-1,1]) = [0.269,0.731]')
# --- independence of external cohorts: no external record duplicates a development record
cl13=set(tuple(r[:13]) for r in raw('cleveland')); key6=lambda r:(r[0],r[1],r[3],r[4],r[7],r[9]); cl6=set(key6(r) for r in raw('cleveland'))
for site in ['hungarian','va']:
    rr=raw(site); d13=sum(tuple(r[:13]) in cl13 for r in rr); d6=sum(key6(r) in cl6 for r in rr)
    ok(d13==0 and d6==0, f'{site}: records duplicating a Cleveland record: {d13} (13 predictors), {d6} (age, sex, BP, chol, max HR, ST depression)')


In [ ]:
%%writefile a2_quantum.py
"""Independent audit 2: re-implement the circuits in plain NumPy and compare with the PennyLane code used."""
import sys, numpy as np, pickle, json

from common import *; from quantum import VQC, QSVM, ent_pairs
import pennylane as qml
ok=lambda c,m: print(('PASS ' if c else 'FAIL ')+m)
def ry(t): c,s=np.cos(t/2),np.sin(t/2); return np.array([[c,-s],[s,c]],complex)
def rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]])
H=np.array([[1,1],[1,-1]])/np.sqrt(2)
def apply1(psi,U,q,n):
    psi=psi.reshape([2]*n); psi=np.moveaxis(psi,q,0); psi=np.tensordot(U,psi,1); return np.moveaxis(psi,0,q).reshape(-1)
def cnot(psi,c,t,n):
    psi=psi.reshape([2]*n).copy(); idx=[slice(None)]*n; idx[c]=1
    sub=psi[tuple(idx)]; ax=t if t<c else t-1; psi[tuple(idx)]=np.flip(sub,axis=ax); return psi.reshape(-1)
def zz(psi,phi,i,j,n):
    psi=psi.reshape([2]*n).copy()
    for a in (0,1):
        for b in (0,1):
            idx=[slice(None)]*n; idx[i]=a; idx[j]=b; s=1 if a==b else -1
            psi[tuple(idx)]*=np.exp(-1j*phi/2*s)
    return psi.reshape(-1)
def expZ0(psi,n):
    p=np.abs(psi.reshape([2]*n))**2; return p[0].sum()-p[1].sum()
def np_vqc(x,par,nq=4,L=2,ent='circular',reup=True,te=False,enc='angle'):
    psi=np.zeros(2**nq,complex); psi[0]=1; nv=2*nq*L
    for l in range(L):
        if reup or l==0:
            if enc=='angle':
                for i in range(nq):
                    a,b=(par[nv+l*2*nq+2*i],par[nv+l*2*nq+2*i+1]) if te else (1,1)
                    psi=apply1(psi,ry(a*x[i]),i,nq); psi=apply1(psi,rz(b*x[i]),i,nq)
            elif enc=='iqp':
                for i in range(nq): psi=apply1(psi,H,i,nq); psi=apply1(psi,rz(x[i]),i,nq)
                for i,j in ent_pairs(nq,'full'): psi=zz(psi,x[i]*x[j],i,j,nq)
        for i in range(nq):
            psi=apply1(psi,ry(par[l*2*nq+2*i]),i,nq); psi=apply1(psi,rz(par[l*2*nq+2*i+1]),i,nq)
        for i,j in ent_pairs(nq,ent): psi=cnot(psi,i,j,nq)
    return expZ0(psi,nq).real
rng=np.random.RandomState(3); X=rng.uniform(0,np.pi,(6,8))
cases=[dict(L=2),dict(L=3),dict(L=2,trainable_enc=True),dict(L=1),dict(L=2,entangle='linear'),dict(L=2,entangle='full'),
       dict(L=2,reupload=False),dict(L=2,encoding='iqp'),dict(L=2,nq=6),dict(L=2,nq=8)]
for kw in cases:
    m=VQC(**kw); p=rng.uniform(-np.pi,np.pi,m.n_par); nq=m.nq
    a=m.raw(X[:,:nq],p)
    b=np.array([np_vqc(x[:nq],p,nq=nq,L=m.L,ent=m.entangle,reup=m.reupload,te=m.trainable_enc,enc=m.encoding) for x in X])
    ok(np.abs(a-b).max()<1e-10, f'circuit {kw}: PennyLane vs NumPy max |diff| {np.abs(a-b).max():.1e}')
# amplitude encoding: check normalised state
m=VQC(L=2,encoding='amplitude'); p=rng.uniform(-np.pi,np.pi,m.n_par); x=rng.randn(16)
psi=x/np.linalg.norm(x)+0j
for l in range(2):
    for i in range(4): psi=apply1(psi,ry(p[l*8+2*i]),i,4); psi=apply1(psi,rz(p[l*8+2*i+1]),i,4)
    for i,j in ent_pairs(4,'circular'): psi=cnot(psi,i,j,4)
ok(abs(m.raw(x[None,:],p)[0]-expZ0(psi,4).real)<1e-10,'amplitude-encoded circuit matches NumPy')
# original notebook VQC-DR circuit (verbatim structure, per patient) vs broadcast implementation
dev=qml.device('default.qubit',wires=4)
@qml.qnode(dev)
def vqc_dr_nb(x,params):
    for l in range(2):
        for i in range(4): qml.RY(x[i],wires=i); qml.RZ(x[i],wires=i)
        for i in range(4):
            qml.RY(params[l*4*2+i*2],wires=i); qml.RZ(params[l*4*2+i*2+1],wires=i)
        for i in range(4): qml.CNOT(wires=[i,(i+1)%4])
    return qml.expval(qml.PauliZ(0))
p=rng.uniform(-np.pi,np.pi,16); m=VQC(L=2)
ok(np.abs(m.raw(X[:,:4],p)-np.array([float(vqc_dr_nb(x[:4],p)) for x in X])).max()<1e-12,'broadcast VQC-DR identical to first-version notebook circuit')
# parameter-shift exactness vs central finite differences
m=VQC(L=2); x=X[:1,:4]; p=rng.uniform(-np.pi,np.pi,16); e=1e-6; mx=0
for k in range(16):
    pp=p.copy(); pp[k]+=np.pi/2; pm=p.copy(); pm[k]-=np.pi/2; ps=(m.raw(x,pp)-m.raw(x,pm))[0]/2
    pp=p.copy(); pp[k]+=e; pm=p.copy(); pm[k]-=e; fd=(m.raw(x,pp)-m.raw(x,pm))[0]/(2*e); mx=max(mx,abs(ps-fd))
ok(mx<1e-6, f'parameter-shift gradient equals numerical derivative (max diff {mx:.1e})')
# training gradient of BCE loss: check one Adam step's gradient against numerical loss derivative
cl,hu=load('cleveland'),load('hungarian'); Xc,y=xy(cl); Xe,ye=xy(hu)
prep=quantum_prep(4).fit(Xc); A=prep.transform(Xc); Ae=prep.transform(Xe)
m=VQC(L=2); p=np.random.RandomState(42).uniform(-np.pi,np.pi,16)
def loss(p):
    pr=np.clip(1/(1+np.exp(-m.raw(A,p))),1e-7,1-1e-7); return -np.mean(y*np.log(pr)+(1-y)*np.log(1-pr))
raw=m.raw(A,p); pr=1/(1+np.exp(-raw)); dL=-(y/pr-(1-y)/(1-pr))*pr*(1-pr)/len(y)
g=[np.dot(dL,(m.raw(A,p+np.eye(16)[k]*np.pi/2)-m.raw(A,p-np.eye(16)[k]*np.pi/2))/2) for k in range(16)]
gn=[(loss(p+np.eye(16)[k]*1e-5)-loss(p-np.eye(16)[k]*1e-5))/2e-5 for k in range(16)]
ok(np.abs(np.array(g)-gn).max()<1e-6, f'BCE gradient (chain rule x parameter shift) matches numerical loss gradient ({np.abs(np.array(g)-gn).max():.1e})')
# determinism: refit final VQC-TE and QSVM and compare with saved predictions
PB=pickle.load(open('pred_b.pkl','rb'))
te=VQC(L=2,trainable_enc=True,lr=0.06).fit(A,y)
ok(np.abs(te.predict_proba(Ae)-PB['VQC-TE']['ext']).max()<1e-9,'VQC-TE refit reproduces saved external predictions exactly')
q=QSVM().fit(A,y)
# independent kernel via NumPy ZZ map
def zzstate(x,nq=4,reps=2):
    psi=np.zeros(16,complex); psi[0]=1
    for _ in range(reps):
        for i in range(nq): psi=apply1(psi,H,i,nq); psi=apply1(psi,rz(2*x[i]),i,nq)
        for i,j in ent_pairs(nq,'full'):
            psi=cnot(psi,i,j,nq); psi=apply1(psi,rz(2*(np.pi-x[i])*(np.pi-x[j])),j,nq); psi=cnot(psi,i,j,nq)
    return psi
S=np.array([zzstate(a) for a in A]); K=np.abs(S.conj()@S.T)**2
Y=np.where(y==1,1.,-1.); kta=np.sum(K*np.outer(Y,Y))/(np.linalg.norm(K)*len(y))
ok(abs(kta-q.kta_)<1e-9 and np.linalg.eigvalsh(K).min()>-1e-10, f'QSVM fidelity kernel reproduced independently: KTA {kta:.4f}, min eigenvalue {np.linalg.eigvalsh(K).min():.1e}')
ok(np.abs(q.predict_proba(Ae)-PB['QSVM']['ext']).max()<1e-9,'QSVM refit reproduces saved predictions')
# noise model: p=0 density-matrix result equals statevector; readout & channel sanity
dr=VQC(L=2).fit(A,y)
for p2 in [0.0]:
    mn=VQC(L=2,noise=dict(p1=0,p2=0,gamma=0,readout=0,shots=None)); mn.params_=dr.params_
    ok(np.abs(mn.raw(Ae[:50])-dr.raw(Ae[:50])).max()<1e-10,'noise model with p=0 equals noiseless statevector output')
mn=VQC(L=2,noise=dict(p1=0,p2=0,gamma=0,readout=0.1,shots=None)); mn.params_=dr.params_
ok(np.abs(mn.raw(Ae[:20])-0.8*dr.raw(Ae[:20])).max()<1e-10,'symmetric readout error r scales <Z> by (1-2r)')
# convex-mixture claim: AUC invariant
from sklearn.metrics import roc_auc_score
pe=dr.predict_proba(Ae); ok(all(abs(roc_auc_score(ye,(1-p)*pe+p*0.5)-roc_auc_score(ye,pe))<1e-12 for p in (0.05,0.2,0.5)),'convex mixture leaves AUC unchanged')


In [ ]:
%%writefile a3_secondary.py
"""Independent audit 3: conformal, DCA, subgroups, paired differences, gradient variance, learning-curve design, ORs."""
import sys, numpy as np, pickle, json

from common import *
from sklearn.model_selection import train_test_split
ok=lambda c,m: print(('PASS ' if c else 'FAIL ')+m)
A=json.load(open('res_a.json')); B=json.load(open('res_b.json'))
D=json.load(open('res_d.json')); G=json.load(open('res_g.json'))
PA=pickle.load(open('pred_a.pkl','rb')); PB=pickle.load(open('pred_b.pkl','rb'))
cl,hu,va=load('cleveland'),load('hungarian'),load('va'); X,y=xy(cl); Xe,ye=xy(hu); Xv,yv=xy(va)
from xgboost import XGBClassifier
# --- conformal: independent LAC implementation
def lac(Xtr,ytr,Xte,seed=42):
    Xp,Xc,yp,yc=train_test_split(Xtr,ytr,test_size=0.25,stratify=ytr,random_state=seed)
    m=make_pipe(XGBClassifier(n_estimators=200,max_depth=4,learning_rate=0.1,eval_metric='logloss',random_state=42,verbosity=0,n_jobs=1)).fit(Xp,yp)
    pc=m.predict_proba(Xc); s=1-pc[np.arange(len(yc)),yc]; n=len(s)
    q=np.sort(s)[int(np.ceil((n+1)*0.9))-1]
    pt=m.predict_proba(Xte); return (1-pt)<=q
for nm,(Xt,yt) in {'external':(Xe,ye),'va':(Xv,yv)}.items():
    S=lac(X,y,Xt); cov=S[np.arange(len(yt)),yt].mean(); sz=S.sum(1).mean(); c1=S[yt==1,1].mean()
    r=D['conformal'][nm]
    ok(abs(cov-r['coverage'])<1e-9 and abs(sz-r['size'])<1e-9 and abs(c1-r['cov_y1'])<1e-9, f'conformal {nm}: independent LAC coverage {cov:.3f}, size {sz:.2f}, CAD+ {c1:.3f} (reported {r["coverage"]:.3f},{r["size"]:.2f},{r["cov_y1"]:.3f})')
cv=[]
for tr,te in PB['folds']:
    S=lac(X[tr],y[tr],X[te]); cv.append(S[np.arange(len(te)),y[te]].mean())
ok(abs(np.mean(cv)-D['conformal']['internal_cv']['coverage'])<1e-9, f'conformal internal CV coverage {np.mean(cv):.3f}')
# --- DCA recompute
t=np.array(D['dca']['thresholds']); prev=ye.mean()
for k in ['Logistic regression','VQC-TE','VQC-3L']:
    p=PA[k]['ext'] if k in PA else PB[k]['ext']
    nb=[((p>=u)&(ye==1)).sum()/294-((p>=u)&(ye==0)).sum()/294*u/(1-u) for u in t]
    ok(np.allclose(nb,D['dca']['net_benefit'][k]), f'DCA {k} recomputed')
    print('INFO', k, 'net benefit by threshold', dict(zip(t.round(2), np.round(nb,3))))
print('INFO treat-all', dict(zip(t.round(2),np.round(D['dca']['net_benefit']['treat_all'],3))))
# --- subgroups counts
w=hu.sex.values==0; ok(w.sum()==81 and ye[w].sum()==12, f'Hungarian women {w.sum()} with {ye[w].sum()} events')
a55=hu.age.values>=55; ok(a55.sum()==58 and ye[a55].sum()==26, 'Hungarian age>=55: 58 patients, 26 events')
from sklearn.metrics import roc_auc_score
ok(abs(roc_auc_score(ye[w],PB['VQC-TE']['ext'][w])-D['subgroups']['Women']['VQC-TE']['AUC'])<1e-12,'subgroup AUC (women, VQC-TE) recomputed')
# --- paired differences point estimates
for k,v in G.items():
    q,c=k.replace(' [VA]','').split(' vs ')
    key='va' if '[VA]' in k else 'ext'; yy=yv if key=='va' else ye
    pq=PB[q][key]; pc=PA[c][key] if c in PA else PB[c][key]
    d=roc_auc_score(yy,pq)-roc_auc_score(yy,pc)
    if abs(d-v[0])>1e-12 or not (v[1]<=d<=v[2]): ok(False,'paired '+k)
ok(True,f'all {len(G)} paired-difference point estimates equal AUC differences and lie inside their CIs')
print('INFO VQC-TE vs LR [VA]',np.round(G['VQC-TE vs Logistic regression [VA]'],3),' vs LR-PCA4 [VA]',np.round(G['VQC-TE vs LR-PCA4 [VA]'],3),' vs RF [VA]',np.round(G['VQC-TE vs Random forest [VA]'],3))
# --- gradient variance spot check at 4 qubits, different seed: should agree within sampling error
from quantum import VQC
prep=quantum_prep(4).fit(X); Aq=prep.transform(X); m=VQC(nq=4,L=2); rng=np.random.RandomState(11); G4=[]
for _ in range(200):
    p=rng.uniform(-np.pi,np.pi,16); x=Aq[rng.randint(303)][None,:]
    G4.append([(m.raw(x,p+np.pi/2*np.eye(16)[j])[0]-m.raw(x,p-np.pi/2*np.eye(16)[j])[0])/2 for j in range(16)])
v4=np.var(np.array(G4),axis=0,ddof=1).mean(); r=D['gradvar']['per_qubit']['4']['mean_var_all']
ok(abs(v4-r)/r<0.25, f'gradient variance at 4 qubits, independent seed: {v4:.4f} vs reported {r:.4f}')
gv=D['gradvar']; q=np.array([2,4,6,8]); vv=np.array([gv['per_qubit'][str(k)]['mean_var_all'] for k in q])
ok(abs(np.polyfit(q,np.log10(vv),1)[0]-gv['slope_log10_all'])<1e-12 and abs(vv[0]/vv[-1]-gv['ratio_2_8_all'])<1e-9, f'slope {gv["slope_log10_all"]:.3f}/qubit, ratio {vv[0]/vv[-1]:.1f}, per-qubit factor {10**gv["slope_log10_all"]:.3f}')
# --- learning-curve design: identical stratified sub-samples
for n in [30,60,120,240]:
    for r_ in range(5):
        idx,_=train_test_split(np.arange(303),train_size=n,stratify=y,random_state=100+r_)
        assert len(idx)==n and abs(y[idx].mean()-y.mean())<0.04
ok(True,'learning-curve sub-samples: correct sizes and stratified (same idx reused for every model by construction of the loop)')
LC=json.load(open('res_c.json'))
for k in ['Logistic regression','VQC-TE','VQC-DR','LR-PCA4']:
    full=A['classical'][k]['ext']['AUC'] if k in A['classical'] else B[k]['ext']['AUC']
    ok(abs(LC[k]['303'][0]-full)<1e-9, f'learning curve n=303 point for {k} equals main analysis ({full:.4f})')
mq=max(np.mean(LC[q_]['60']) for q_ in ['VQC-DR','VQC-3L','VQC-TE']); mc=max(np.mean(LC[c]['60']) for c in LC if not c.startswith('VQC'))
print(f'INFO n=60 best quantum {mq:.4f} vs best classical {mc:.4f}: margin {mq-mc:+.4f}')
for n in ['30','60','120','240']:
    sds={k:np.std(v[n],ddof=1) for k,v in LC.items() if not k.endswith('_time') and not (k=='LightGBM' and n=='30')}
    print('INFO SD ranking n='+n, sorted(sds,key=sds.get,reverse=True)[:4], [round(sds[k],3) for k in sorted(sds,key=sds.get,reverse=True)[:4]])
# --- unadjusted ORs, independent (numpy Newton)
def logit(xv,yv_):
    Xd=np.c_[np.ones(len(xv)),xv]; b=np.zeros(2)
    for _ in range(50):
        p=1/(1+np.exp(-Xd@b)); W=p*(1-p); b+=np.linalg.solve(Xd.T@(Xd*W[:,None]),Xd.T@(yv_-p))
    return b[1]
mx=0
for c,o in A['unadjusted_or'].items():
    s=cl[[c,'target']].dropna(); xv=s[c].values.astype(float)
    if o['per']=='SD': xv=(xv-xv.mean())/xv.std()
    mx=max(mx,abs(np.exp(logit(xv,s.target.values))-o['OR']))
ok(mx<1e-6, f'unadjusted ORs recomputed independently (max diff {mx:.1e})')


In [ ]:
import subprocess, sys
if RUN_AUDIT:
    log = ''
    for f in ['a1_data_metrics.py', 'a2_quantum.py'] + (['a3_secondary.py'] if RUN_LEARNING_CURVES else []):
        r = subprocess.run([sys.executable, f], capture_output=True, text=True)
        log += r.stdout; print(r.stdout[-6000:]);
        assert r.returncode == 0, r.stderr[-3000:]
    n_pass, n_fail = log.count('PASS'), log.count('FAIL')
    print(f'\nAUDIT SUMMARY: {n_pass} PASS, {n_fail} FAIL'); assert n_fail == 0
    open('audit_log.txt', 'w').write(log)

## 7. Package the outputs

In [ ]:
import shutil, glob
os.makedirs('outputs', exist_ok=True)
for f in glob.glob('res_*.json') + glob.glob('audit_log.txt'): shutil.copy(f, 'outputs')
shutil.copytree('figures', 'outputs/figures', dirs_exist_ok=True)
shutil.copytree('tables', 'outputs/tables', dirs_exist_ok=True)
print('outputs:', sorted(os.listdir('outputs')), '| figures:', len(os.listdir('outputs/figures')), '| table files:', len(os.listdir('outputs/tables')))
shutil.make_archive('qml_heart_reproduction_outputs', 'zip', 'outputs')
print('created', os.path.abspath('qml_heart_reproduction_outputs.zip'))
try:
    from google.colab import files; files.download('qml_heart_reproduction_outputs.zip')
except Exception:
    pass